# 01 — Direct risk sensitivity: synchronized manuscript Table 2

This notebook recomputes the 30-case Model A/Model B grid and updates the manuscript table
`braess_modelB_direct_eta_table_full.tex` from those results. It is based on the uploaded
`01_direct_eta_risk_sensitivity_VERIFIED(1).ipynb`; the benchmark parameters, equilibrium
solvers, KKT-compatible law selection, and numerical tolerances are unchanged.

## Run and use the updated table

Choose **Restart Kernel and Run All**. All definitions and benchmark inputs are embedded;
no other notebook, Python module, old CSV, or old LaTeX table is needed to compute results.
For missing packages, run this in a separate cell:

```python
%pip install numpy scipy pandas matplotlib
```

The **Output location and manuscript target** cell sets `PAPER_TABLE_PATH`. Its default is
`braess_modelB_direct_eta_table_full.tex` in the notebook working directory. Set this variable
to the exact table file used by your LaTeX project when that file is in another folder.
Notebook execution does not update a remote Overleaf project automatically.

Each run also creates a new `verified_braess_outputs/run_<UTC timestamp>/` directory.
The paper table is updated only after the 30 cases, source checks, CSV/LaTeX read-back checks,
and deliberate-corruption tests pass. An existing copy of the same table is backed up in
`01_table2/publication_audit/` before replacement. A different manuscript or a file with
another table label is not overwritten. The notebook displays direct links to the exact
updated table, full-precision source CSV, and validation report.

## What is synchronized

Every table cell comes from the same current solved records: both flow vectors, both
reservation costs, pure-tail and effective-law distances, flow distance, the Model A VI gap
at the Model B solution, and the difference in active-path counts. No old table values are
used to fill new cells. The original 16-column layout, table label, alpha-group spacing,
and display precision are retained; small nonzero flow distances and VI gaps use scientific
notation. The caption clarifies the KKT-compatible selector and the eta=0 reporting convention.

## Checks and scope

The existing mandatory numerical regressions are retained. All 30 grid cases are recomputed,
with Model A solved independently. Model B costs and reservation costs use its selected
KKT-compatible common law, not a re-sorted value-only selector. The exporter independently
recomputes published diagnostics from the solved records, checks every one of the 450 numeric
LaTeX cells against the source CSV, and verifies that the manuscript copy and run copy are
byte-for-byte identical. Corrupted numerical source, CSV, and LaTeX copies must be rejected.

These checks use the explicitly reported floating-point tolerances. They are not a proof in
exact arithmetic or a validation of untested parameters, other networks, or empirical assumptions.
At eta=0, the pure-tail law is a value-only convention and does not affect equilibrium.


## Definitions

The following sections contain the full common implementation. No external source file is required.

In [1]:
"""Verified finite-support Braess TSUE experiments.

Data constructors are retained from the four user-uploaded notebooks.
The risk solvers and all law/KKT/certificate recovery are rewritten.
No commercial solver, local data file, CVXPY, or notebook-to-notebook import is needed.
"""
from __future__ import annotations

from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Sequence
import itertools
import json
import math
import platform
import sys
import time
import hashlib
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize, linprog, brentq, least_squares



### Original benchmark data (no parameter rounding)

In [2]:
# Original benchmark data (no parameter rounding)


Q_DEMAND = 80.0

THETA = 0.60

R0 = 1.0

BPR_BETA = 0.15

BPR_GAMMA = 4.0

ALPHA_MAIN = 0.90

ETA_MAIN = 0.50

LINKS = ("AB", "AC", "BC", "BD", "CD")

LINK_LABELS = {
    "AB": "A→B",
    "AC": "A→C",
    "BC": "B→C",
    "BD": "B→D",
    "CD": "C→D",
}

BASE_LINK_PARAMS = {
    # link: (free-flow time t0, capacity c, additive delay Delta)
    # v5 shortcut-active calibration: P3=A-B-C-D is attractive in ordinary
    # conditions but vulnerable under the low-probability shortcut incident.
    "AB": (3.0, 130.0, 0.0),
    "AC": (7.5, 85.0, 0.0),
    "BC": (0.8, 120.0, 0.0),
    "BD": (14.0, 65.0, 0.0),
    "CD": (8.0, 110.0, 0.0),
}

SCENARIOS = ("Normal", "Upper incident", "Lower incident", "Shortcut incident", "System-wide")

SCENARIO_PROBS = np.array([0.55, 0.18, 0.17, 0.06, 0.04], dtype=float)

@dataclass(frozen=True)
class LinkScenarioParams:
    t0: float
    capacity: float
    delay: float

@dataclass(frozen=True)
class PathSet:
    name: str
    path_names: tuple[str, ...]
    incidence: np.ndarray  # shape: n_paths x n_links

    @property
    def n_paths(self) -> int:
        return len(self.path_names)

    def validate(self) -> None:
        if self.incidence.shape != (self.n_paths, len(LINKS)):
            raise ValueError(f"Invalid incidence shape for {self.name}: {self.incidence.shape}.")
        if not np.all((self.incidence == 0) | (self.incidence == 1)):
            raise ValueError("Path-link incidence must be binary.")
        if np.any(self.incidence.sum(axis=1) <= 0):
            raise ValueError("Every path must use at least one link.")

@dataclass(frozen=True)
class BraessInstance:
    name: str
    scenario_names: tuple[str, ...]
    probabilities: np.ndarray
    link_params: dict[str, dict[str, LinkScenarioParams]]

@dataclass(frozen=True)
class SolverConfig:
    q: float = Q_DEMAND
    theta: float = THETA
    r0: float = R0
    alpha: float = ALPHA_MAIN
    eta: float = ETA_MAIN
    bpr_beta: float = BPR_BETA
    bpr_gamma: float = BPR_GAMMA

def base_path_set() -> PathSet:
    incidence = np.array(
        [
            [1, 0, 0, 1, 0],  # P1 = A-B-D
            [0, 1, 0, 0, 1],  # P2 = A-C-D
            [1, 0, 1, 0, 1],  # P3 = A-B-C-D
        ],
        dtype=float,
    )
    return PathSet(
        name="Base-3 paths",
        path_names=("P1: A→B→D", "P2: A→C→D", "P3: A→B→C→D"),
        incidence=incidence,
    )

def build_common_tail_instance() -> BraessInstance:
    """Scenario-ordered instance satisfying the common-tail condition."""
    severity_schedule = {
        "Normal": (1.00, 1.00, 0.0),
        "Upper incident": (1.04, 0.92, 1.5),
        "Lower incident": (1.08, 0.84, 3.5),
        "Shortcut incident": (1.12, 0.76, 6.0),
        "System-wide": (1.18, 0.65, 10.0),
    }
    params: dict[str, dict[str, LinkScenarioParams]] = {}
    for scenario in SCENARIOS:
        t_mult, cap_mult, delay_add = severity_schedule[scenario]
        params[scenario] = {}
        for link, (t0, capacity, delay) in BASE_LINK_PARAMS.items():
            params[scenario][link] = LinkScenarioParams(
                t0=t0 * t_mult,
                capacity=capacity * cap_mult,
                delay=delay + delay_add,
            )
    return BraessInstance(
        name="Common-tail Braess",
        scenario_names=SCENARIOS,
        probabilities=SCENARIO_PROBS.copy(),
        link_params=params,
    )

def build_route_local_instance() -> BraessInstance:
    """Non-common-tail instance with a shortcut-attractive but tail-risky P3.

    The normal-state parameters make P3=A-B-C-D a plausible shortcut.
    Route-local incidents can make Model A and Model B select different tails.
    No monotone change in a particular route share is assumed.
    """
    params: dict[str, dict[str, LinkScenarioParams]] = {}
    for scenario in SCENARIOS:
        params[scenario] = {}
        for link, (t0, capacity, delay) in BASE_LINK_PARAMS.items():
            t_mult, cap_mult, delay_add = 1.0, 1.0, 0.0
            if scenario == "Upper incident":
                # Primarily penalizes P1 through BD and mildly penalizes P3 through AB.
                if link == "BD":
                    t_mult, cap_mult, delay_add = 1.18, 0.55, 16.0
                elif link == "AB":
                    t_mult, cap_mult, delay_add = 1.08, 0.78, 4.0
                else:
                    t_mult, cap_mult, delay_add = 1.02, 0.96, 0.3
            elif scenario == "Lower incident":
                # Primarily penalizes P2 through AC and mildly penalizes P3 through CD.
                if link == "AC":
                    t_mult, cap_mult, delay_add = 1.14, 0.62, 10.0
                elif link == "CD":
                    t_mult, cap_mult, delay_add = 1.08, 0.75, 4.0
                else:
                    t_mult, cap_mult, delay_add = 1.02, 0.96, 0.3
            elif scenario == "Shortcut incident":
                # Makes P3 attractive on average but risky in the upper tail.
                if link == "BC":
                    t_mult, cap_mult, delay_add = 1.25, 0.35, 24.0
                else:
                    t_mult, cap_mult, delay_add = 1.01, 0.98, 0.2
            elif scenario == "System-wide":
                t_mult, cap_mult, delay_add = 1.22, 0.58, 10.0

            params[scenario][link] = LinkScenarioParams(
                t0=t0 * t_mult,
                capacity=capacity * cap_mult,
                delay=delay + delay_add,
            )
    return BraessInstance(
        name="Route-local tail Braess",
        scenario_names=SCENARIOS,
        probabilities=SCENARIO_PROBS.copy(),
        link_params=params,
    )

def validate_instance(instance: BraessInstance) -> None:
    if not np.isclose(instance.probabilities.sum(), 1.0, atol=1e-12):
        raise ValueError("Scenario probabilities must sum to one.")
    if np.any(instance.probabilities < 0):
        raise ValueError("Scenario probabilities must be nonnegative.")
    for scenario in instance.scenario_names:
        if scenario not in instance.link_params:
            raise ValueError(f"Missing scenario {scenario}.")
        for link in LINKS:
            prm = instance.link_params[scenario][link]
            if prm.t0 <= 0 or prm.capacity <= 0 or prm.delay < 0:
                raise ValueError(f"Invalid parameters for scenario={scenario}, link={link}: {prm}.")

SCENARIO_VECTORS = np.array(
    [
        [0.00, 0.00, 0.00],
        [1.00, 0.00, 0.00],
        [0.00, 1.00, 0.00],
        [0.00, 0.00, 1.00],
        [0.85, 0.85, 0.85],
    ],
    dtype=float,
)

def instance_with_probabilities(instance: BraessInstance, probabilities: Sequence[float], name: str) -> BraessInstance:
    probs = np.asarray(probabilities, dtype=float)
    if probs.shape[0] != len(instance.scenario_names):
        raise ValueError("Probability vector length does not match scenario count.")
    if np.any(probs < 0.0) or not np.isclose(probs.sum(), 1.0, atol=1e-10):
        raise ValueError("Probabilities must be nonnegative and sum to one.")
    return BraessInstance(name=name, scenario_names=instance.scenario_names, probabilities=probs, link_params=instance.link_params)

def build_reliable_bc_upgrade(instance: BraessInstance) -> BraessInstance:
    """Return a before/after instance with a more reliable B-C link.

    The upgraded B-C link is deliberately less attractive in the normal state
    but much less vulnerable in shortcut and system-wide severe states.
    """
    params: dict[str, dict[str, LinkScenarioParams]] = {}
    for scenario in instance.scenario_names:
        params[scenario] = {}
        for link, old in instance.link_params[scenario].items():
            if link != "BC":
                params[scenario][link] = old
                continue
            if scenario == "Normal":
                params[scenario][link] = LinkScenarioParams(t0=1.76, capacity=78.0, delay=4.00)
            elif scenario == "Shortcut incident":
                params[scenario][link] = LinkScenarioParams(t0=1.20, capacity=110.0, delay=3.00)
            elif scenario == "System-wide":
                params[scenario][link] = LinkScenarioParams(t0=1.40, capacity=100.0, delay=5.00)
            else:
                params[scenario][link] = LinkScenarioParams(t0=1.20, capacity=95.0, delay=1.50)
    return BraessInstance(
        name="Reliable B-C upgrade",
        scenario_names=instance.scenario_names,
        probabilities=instance.probabilities.copy(),
        link_params=params,
    )

def scenario_distance_matrix() -> np.ndarray:
    return np.linalg.norm(SCENARIO_VECTORS[:, None, :] - SCENARIO_VECTORS[None, :, :], axis=2)



### Numerical tolerances, output records, and cost derivatives

In [3]:
# Numerical tolerances, output records, and cost derivatives

CORE_VERSION = "braess-kkt-joint-risk-envelope-v1"
TRAIN_COUNTS = np.array([6, 2, 2, 1, 0], dtype=int)
TRAIN_LAW = TRAIN_COUNTS / TRAIN_COUNTS.sum()
TEST_LAW = np.array([0.48, 0.17, 0.16, 0.12, 0.07], dtype=float)

@dataclass(frozen=True)
class CheckTolerances:
    """Acceptance thresholds, not statements of exact arithmetic.

    Risk and certificate tolerances are absolute per unit of OD demand.
    KKT is in path-time units; kernel error is normalized by demand.
    """
    probability: float = 2e-9
    flow_feasibility: float = 2e-8
    kkt: float = 1e-6
    kernel_relative: float = 2e-7
    risk_per_demand: float = 5e-8
    global_gap_per_demand: float = 1e-6
    a_vi_relative: float = 2e-9
    a_fixed_point_relative: float = 2e-9

TOL = CheckTolerances()
LP_OPTIONS = {"primal_feasibility_tolerance": 1e-10,
              "dual_feasibility_tolerance": 1e-10,
              "ipm_optimality_tolerance": 1e-12}

class NumericalValidationError(RuntimeError):
    """Raised before exporting a result that fails an acceptance check."""

@dataclass
class SolvedEquilibrium:
    model: str
    instance: str
    cfg: SolverConfig
    flows: np.ndarray
    path_costs: np.ndarray
    pi: float
    raw_law: np.ndarray | None
    tail_law: np.ndarray | None
    tilted_law: np.ndarray | None
    transport: np.ndarray | None
    objective: float
    risk_value: float
    gamma: float | None
    kappa: float | None
    iterations: int
    elapsed_seconds: float
    checks: dict[str, float | str | bool]
    tail_law_identified: bool = True

    @property
    def shares(self):
        return self.flows / self.cfg.q

    @property
    def active_count(self):
        return int(np.count_nonzero(self.flows > 1e-6))


def checked_probabilities(p, length=None):
    p = np.asarray(p, dtype=float)
    if p.ndim != 1 or (length is not None and len(p) != length):
        raise ValueError("Probability vector has incorrect dimensions.")
    if not np.all(np.isfinite(p)) or np.any(p < 0) or abs(p.sum()-1) > 1e-12:
        raise ValueError("Probabilities must be finite, nonnegative, and sum to one.")
    return p.copy()


def validate_configuration(instance, paths, cfg):
    validate_instance(instance)
    paths.validate()
    checked_probabilities(instance.probabilities, len(instance.scenario_names))
    if not all(np.isfinite(x) for x in (cfg.q,cfg.theta,cfg.r0,cfg.alpha,cfg.eta,cfg.bpr_beta,cfg.bpr_gamma)):
        raise ValueError("All model parameters must be finite.")
    if cfg.q <= 0 or cfg.theta <= 0 or cfg.r0 <= 0:
        raise ValueError("Demand, theta, and r0 must be positive.")
    if not 0 < cfg.alpha < 1 or not 0 <= cfg.eta <= 1:
        raise ValueError("This implementation requires 0 < alpha < 1 and 0 <= eta <= 1.")
    if cfg.bpr_beta < 0 or cfg.bpr_gamma < 1:
        raise ValueError("BPR beta must be nonnegative and exponent >= 1.")


class CostEvaluator:
    """Vectorized original BPR costs, canonical potentials, and derivatives."""
    def __init__(self, instance, paths, cfg):
        validate_configuration(instance, paths, cfg)
        self.instance, self.paths, self.cfg = instance, paths, cfg
        self.A = np.asarray(paths.incidence, dtype=float)
        self.t0 = np.array([[instance.link_params[s][a].t0 for a in LINKS]
                            for s in instance.scenario_names])
        self.cap = np.array([[instance.link_params[s][a].capacity for a in LINKS]
                             for s in instance.scenario_names])
        self.delay = np.array([[instance.link_params[s][a].delay for a in LINKS]
                               for s in instance.scenario_names])
        self.linear = self.t0 + self.delay
        self.poly = self.t0 * cfg.bpr_beta / self.cap**cfg.bpr_gamma

    def values(self, shares):
        """Return L/q, d(L/q)/d(shares)=T, physical link times, and link flows."""
        y = np.asarray(shares, dtype=float)
        x = self.cfg.q * (self.A.T @ y)
        if np.min(x) < -1e-8:
            raise ValueError("Negative link flows outside the permitted roundoff range.")
        x = np.maximum(x, 0.0)
        g = self.cfg.bpr_gamma
        times = self.linear + self.poly * x[None,:]**g
        L = np.sum(self.linear*x[None,:] + self.poly*x[None,:]**(g+1)/(g+1),axis=1)
        return L/self.cfg.q, times@self.A.T, times, x

    def regularizer(self, shares):
        """Return entropy/(theta*q), with gradient with respect to shares."""
        f = self.cfg.q * np.asarray(shares, dtype=float)
        if np.min(f) < -self.cfg.r0:
            raise ValueError("Regularizer evaluated outside its domain.")
        log = np.log1p(f/self.cfg.r0)
        value = np.sum((f+self.cfg.r0)*log-f)/(self.cfg.theta*self.cfg.q)
        return float(value), log/self.cfg.theta


def finite_tail_law(values, probabilities, alpha):
    """A value-valid CVaR tail law; safe at zero probabilities and exact ties.

    This is NOT a Model B equilibrium-gradient recovery method. Stable ordering
    only makes a value-only tail selection reproducible. Model B/DRO laws are
    selected jointly with stationarity by select_kkt_law below.
    """
    v = np.asarray(values,dtype=float)
    p = checked_probabilities(probabilities, len(v))
    if v.ndim != 1 or not np.all(np.isfinite(v)) or not 0 < alpha < 1:
        raise ValueError("Invalid finite CVaR inputs.")
    beta = 1-alpha
    nu = np.zeros_like(p)
    remaining = beta
    for j in np.argsort(-v,kind="stable"):
        take = min(float(p[j]), remaining)
        nu[j] = take
        remaining -= take
        if remaining <= 2*np.finfo(float).eps*beta:
            break
    if abs(nu.sum()-beta) > 2e-14:
        raise NumericalValidationError("CVaR tail mass was not filled.")
    return nu/beta


def finite_cvar(values, probabilities, alpha):
    return float(finite_tail_law(values,probabilities,alpha) @ np.asarray(values))


def ru_cvar_independent(values, probabilities, alpha):
    """Independent value computation: minimize the RU expression at its knots."""
    v=np.asarray(values,dtype=float)
    p=checked_probabilities(probabilities,len(v))
    candidates = np.unique(v[p>0])
    return float(min(t+np.dot(p,np.maximum(v-t,0))/(1-alpha) for t in candidates))


def reservation_and_target(costs, cfg):
    """Stable exact active-set inversion of the truncated kernel (no clipping)."""
    c=np.asarray(costs,dtype=float)
    if c.ndim != 1 or not np.all(np.isfinite(c)):
        raise ValueError("Invalid generalized path costs.")
    order=np.argsort(c,kind="stable")
    cmin=c[order[0]]
    for m in range(1,len(c)+1):
        active=order[:m]
        pi=cmin+(np.log(cfg.q/cfg.r0+m)-np.log(np.exp(-cfg.theta*(c[active]-cmin)).sum()))/cfg.theta
        if m == len(c) or pi <= c[order[m]]:
            target=np.zeros_like(c)
            target[active]=cfg.r0*np.expm1(cfg.theta*(pi-c[active]))
            if np.min(target) < -1e-10 or abs(target.sum()-cfg.q)>1e-9:
                raise NumericalValidationError("Invalid truncated-kernel inversion.")
            return float(pi),target
    raise NumericalValidationError("Could not locate the truncated active set.")


def pathwise_costs(evaluator, shares):
    _,T,_,_=evaluator.values(shares)
    cfg=evaluator.cfg
    p=evaluator.instance.probabilities
    tails=np.vstack([finite_tail_law(T[:,k],p,cfg.alpha) for k in range(T.shape[1])])
    tilts=(1-cfg.eta)*p[None,:]+cfg.eta*tails
    costs=np.einsum("ks,sk->k",tilts,T)
    return costs,tails,tilts


def model_a_diagnostics(evaluator, shares):
    cfg=evaluator.cfg
    c,_,_=pathwise_costs(evaluator,shares)
    f=np.asarray(shares)*cfg.q
    pi,target=reservation_and_target(c,cfg)
    F=c+np.log1p(f/cfg.r0)/cfg.theta
    gap=float(np.dot(F,f)-cfg.q*np.min(F))
    return c,pi,{
        "demand_error":abs(float(f.sum()-cfg.q)),
        "nonnegative_flow_error":max(0.,-float(f.min())),
        "fixed_point_relative":float(np.max(np.abs(target-f))/cfg.q),
        "vi_gap_relative":max(0.,gap)/(1+abs(float(F@f))),
        "vi_gap_absolute":max(0.,gap),
    }


def solve_model_a(instance, paths, cfg, initial_flows=None, relaxation=0.2,
                  tolerance=1e-9, max_iter=80000):
    """Damped fixed point retained for comparable warm-start iteration counts.

    The residual is evaluated at the SAME returned flow and cost vector.
    No unvalidated post-solve target-flow replacement is made.
    """
    ev=CostEvaluator(instance,paths,cfg)
    start=time.perf_counter()
    if initial_flows is None:
        f=np.full(paths.n_paths,cfg.q/paths.n_paths)
    else:
        f=np.asarray(initial_flows,dtype=float).copy()
        if len(f)!=paths.n_paths or np.min(f)<0 or abs(f.sum()-cfg.q)>1e-6:
            raise ValueError("Model A initial flows must be feasible.")
    for iteration in range(1,max_iter+1):
        c,_,_=pathwise_costs(ev,f/cfg.q)
        pi,target=reservation_and_target(c,cfg)
        residual=float(np.max(np.abs(target-f)))
        if residual <= tolerance:
            break
        f=(1-relaxation)*f+relaxation*target
    else:
        raise NumericalValidationError(f"Model A did not converge: residual={residual:.3e}.")
    c,pi,checks=model_a_diagnostics(ev,f/cfg.q)
    if checks["vi_gap_relative"]>TOL.a_vi_relative or checks["fixed_point_relative"]>TOL.a_fixed_point_relative:
        raise NumericalValidationError(f"Model A failed its own VI/kernel checks: {checks}")
    checks["passed"]=True
    return SolvedEquilibrium("Model A",instance.name,cfg,f,c,pi,None,None,None,None,
                            float("nan"),float("nan"),None,None,iteration,
                            time.perf_counter()-start,checks,tail_law_identified=(cfg.eta>0))



### Joint distribution/tail envelope and KKT-compatible selection

In [4]:
# Joint distribution/tail envelope and KKT-compatible selection

class FiniteRiskEnvelope:
    """Independent LP representation of the full mean-CVaR risk.

    Nominal: v >= 0, sum(v)=1, (1-alpha)*v <= p.
    DRO: P >= 0, P 1 = p_observed, <D,P> <= rho,
         q=P.T 1, v>=0, sum(v)=1, (1-alpha)*v <= q.
    Effective law w = (1-eta)*q + eta*v.
    v is normalized tail mass; nu=(1-alpha)*v.
    """
    def __init__(self, probabilities, alpha, eta, rho=None, distances=None):
        self.p=checked_probabilities(probabilities)
        self.m=len(self.p)
        self.alpha=float(alpha)
        self.beta=1-alpha
        self.eta=float(eta)
        self.rho=None if rho is None else float(rho)
        self.observed=np.flatnonzero(self.p>0)
        m=self.m
        if not 0<alpha<1 or not 0<=eta<=1:
            raise ValueError("Invalid risk parameters.")
        if rho is None or rho==0:
            self.robust=False
            self.nz=m
            self.W=eta*np.eye(m)
            self.w0=(1-eta)*self.p
            self.Q=np.zeros((m,m))
            self.q0=self.p.copy()
            self.V=np.eye(m)
            self.Aeq=np.ones((1,m)); self.beq=np.ones(1)
            self.Aub=self.beta*np.eye(m); self.bub=self.p.copy()
            self.distances=None
        else:
            if rho<0 or not np.isfinite(rho):
                raise ValueError("Wasserstein radius must be finite and nonnegative.")
            D=np.asarray(distances,dtype=float)
            if D.shape!=(m,m) or not np.all(np.isfinite(D)) or np.min(D)<0 or not np.allclose(np.diag(D),0,atol=1e-12):
                raise ValueError("Invalid ground-distance matrix.")
            if not np.allclose(D,D.T,atol=1e-12):
                raise ValueError("A symmetric ground metric is required.")
            if np.any(D[:,:,None] > D[:,None,:]+D.T[None,:,:]+1e-10):
                raise ValueError("Ground distances fail the triangle inequality.")
            self.robust=True; self.distances=D
            o=len(self.observed); self.nplan=o*m; self.nz=o*m+m
            self.Q=np.zeros((m,self.nz))
            for i in range(o):self.Q[:,i*m:(i+1)*m]=np.eye(m)
            self.q0=np.zeros(m)
            self.V=np.zeros((m,self.nz));self.V[:,-m:]=np.eye(m)
            self.W=(1-eta)*self.Q+eta*self.V
            self.w0=np.zeros(m)
            self.Aeq=np.zeros((o+1,self.nz)); self.beq=np.r_[self.p[self.observed],1.]
            for i in range(o):self.Aeq[i,i*m:(i+1)*m]=1
            self.Aeq[-1,-m:]=1
            transport_row=np.r_[D[self.observed,:].ravel(),np.zeros(m)]
            self.Aub=np.vstack([transport_row,self.beta*self.V-self.Q])
            self.bub=np.r_[rho,np.zeros(m)]
        self.bounds=[(0.,None)]*self.nz

    def maximize(self, losses):
        l=np.asarray(losses,dtype=float)
        coeff=l@self.W
        res=linprog(-coeff,A_ub=self.Aub,b_ub=self.bub,
                    A_eq=self.Aeq,b_eq=self.beq,bounds=self.bounds,
                    method="highs",options=LP_OPTIONS)
        if not res.success:
            raise NumericalValidationError("Joint risk-envelope LP failed: "+res.message)
        return float(l@self.w0+coeff@res.x),res

    def unpack(self,z):
        q=self.q0+self.Q@z
        v=self.V@z
        w=self.w0+self.W@z
        plan=z[:self.nplan].reshape(len(self.observed),self.m) if self.robust else None
        return q,v,w,plan


def select_kkt_law(envelope,losses,path_times,regularizer_gradient,shares):
    """Select a risk-optimal law compatible with the current flow's KKT system.

    First solve the FULL inner risk LP. Then minimize the maximum stationarity
    error over its near-optimal face. For DRO, choose maximal system-wide mass
    only AFTER enforcing risk optimality and the best stationarity tolerance.
    No independent renormalization, fixed-gamma adversary, or empirical fallback.
    """
    env=envelope
    rstar,primary=env.maximize(losses)
    nz=env.nz
    # Absolute per-demand objective tolerance, scaled modestly by risk magnitude.
    face_tolerance=2e-11*(1+abs(rstar))
    G=path_times.T@env.W
    h=path_times.T@env.w0+regularizer_gradient
    Aeq=np.pad(env.Aeq,((0,0),(0,2)))
    Aub=[np.r_[row,0.,0.] for row in env.Aub]
    bub=list(env.bub)
    risk_coeff=np.asarray(losses)@env.W
    Aub.append(np.r_[-risk_coeff,0.,0.])
    bub.append(-(rstar-np.dot(losses,env.w0)-face_tolerance))
    for k in range(len(shares)):
        # Inactive paths also satisfy F_k >= pi.
        Aub.append(np.r_[-G[k],1.,-1.]);bub.append(h[k])
        if shares[k]>1e-8:
            Aub.append(np.r_[G[k],-1.,-1.]);bub.append(-h[k])
    bounds=env.bounds+[(None,None),(0.,None)]
    objective=np.r_[np.zeros(nz+1),1.]
    res=linprog(objective,A_ub=np.asarray(Aub),b_ub=np.asarray(bub),
                A_eq=Aeq,b_eq=env.beq,bounds=bounds,method="highs",options=LP_OPTIONS)
    if not res.success:
        raise NumericalValidationError("Joint risk/KKT law-selection LP failed: "+res.message)
    emin=float(res.x[-1])
    if env.robust:
        # Last state is System-wide in the original benchmark data. This explicit
        # tie-break is a reporting convention, not a claim of unique raw Q*.
        extra=np.r_[np.zeros(nz+1),1.]
        sec_obj=np.r_[-env.Q[-1],0.,0.]
        sec=linprog(sec_obj,A_ub=np.vstack([Aub,extra]),
                    b_ub=np.r_[bub,emin+2e-9],A_eq=Aeq,b_eq=env.beq,
                    bounds=bounds,method="highs",options=LP_OPTIONS)
        if not sec.success:
            raise NumericalValidationError("KKT-preserving law tie-break failed: "+sec.message)
        res=sec
    q,v,w,plan=env.unpack(res.x[:nz])
    if env.eta==0:
        # eta=0 does not identify v. Report a valid value-only selector explicitly.
        v=finite_tail_law(losses,q,env.alpha)
        w=q.copy()
    return {"risk_max":rstar,"q":q,"tail":v,"tilt":w,"plan":plan,
            "pi_lp":float(res.x[-2]),"min_stationarity":emin,
            "face_tolerance":face_tolerance,"risk_lp":primary}


def minimum_transport_cost(p,q,D):
    """Independent optimal-transport LP, rather than just the selected plan cost."""
    p=checked_probabilities(p)
    q=np.asarray(q,dtype=float)
    obs=np.flatnonzero(p>0);o=len(obs);m=len(p)
    A=np.zeros((o+m,o*m))
    for i in range(o):
        A[i,i*m:(i+1)*m]=1
        A[o:,i*m:(i+1)*m]=np.eye(m)
    res=linprog(np.asarray(D)[obs,:].ravel(),A_eq=A,b_eq=np.r_[p[obs],q],
                bounds=(0,None),method="highs",options=LP_OPTIONS)
    if not res.success:
        raise NumericalValidationError("Independent Wasserstein-distance LP failed: "+res.message)
    return float(res.fun)


def certify_model_b(evaluator,shares,envelope,selection,master_value,master_feasibility):
    """A posteriori convex global bound plus separate KKT, CVaR, and OT checks.

    For any feasible selected w, let F = grad_f[w.L(f)+H(f)]. Then
      LB = w.L(f)+H(f) - [F.f-q*min(F)],
      UB = max_w w.L(f)+H(f).
    Thus UB-LB = inner risk gap + simplex VI gap. The LP and floating-point
    feasibility tolerances are reported; this is not interval arithmetic.
    """
    cfg=evaluator.cfg
    l,T,_,_=evaluator.values(shares)
    H,dH=evaluator.regularizer(shares)
    s=selection
    q,v,w=s["q"],s["tail"],s["tilt"]
    f=cfg.q*np.asarray(shares)
    c=T.T@w
    F=c+dH
    active=f>1e-6
    pi=float(np.mean(F[active]))
    kkt=max(float(np.max(np.abs(F[active]-pi))),
            max(0.,pi-float(np.min(F[~active]))) if np.any(~active) else 0.)
    pi_kernel,target=reservation_and_target(c,cfg)
    risk_selected=float(w@l)
    cvar_ru=ru_cvar_independent(l,q,cfg.alpha)
    risk_under_q=(1-cfg.eta)*float(q@l)+cfg.eta*cvar_ru
    vi=max(0.,float(F@shares-np.min(F)))
    UB=s["risk_max"]+H
    LB=risk_selected+H-vi
    checks={
        "demand_error":abs(float(f.sum()-cfg.q)),
        "nonnegative_flow_error":max(0.,-float(f.min())),
        "probability_sum_error":max(abs(float(q.sum()-1)),abs(float(v.sum()-1)),abs(float(w.sum()-1))),
        "probability_nonnegativity_error":max(0.,-float(min(q.min(),v.min(),w.min()))),
        "tail_capacity_violation":max(0.,float(np.max((1-cfg.alpha)*v-q))),
        "tail_value_error_per_demand":abs(float(v@l)-cvar_ru),
        "tilt_identity_error":float(np.max(np.abs(w-((1-cfg.eta)*q+cfg.eta*v)))),
        "inner_risk_gap_per_demand":abs(float(s["risk_max"]-risk_selected)),
        "raw_law_risk_gap_per_demand":abs(float(s["risk_max"]-risk_under_q)),
        "kkt_stationarity":kkt,
        "pi_kernel_error":abs(pi-pi_kernel),
        "pi_lp_error":abs(pi-s["pi_lp"]),
        "kernel_relative_error":float(np.max(np.abs(target-f))/cfg.q),
        "vi_gap_per_demand":vi,
        "global_gap_per_demand":float(UB-LB),
        "lower_bound":float(cfg.q*LB),
        "upper_bound":float(cfg.q*UB),
        "master_objective_error_per_demand":abs(float(master_value-UB)),
        "master_feasibility_error":float(master_feasibility),
        "law_selection_face_tolerance":float(s["face_tolerance"]),
    }
    if envelope.robust:
        plan=s["plan"]
        checks.update({
            "transport_row_error":float(np.max(np.abs(plan.sum(axis=1)-envelope.p[envelope.observed]))),
            "transport_column_error":float(np.max(np.abs(plan.sum(axis=0)-q))),
            "transport_nonnegativity_error":max(0.,-float(plan.min())),
            "transport_cost":float(np.sum(plan*envelope.distances[envelope.observed,:])),
            "wasserstein_distance":minimum_transport_cost(envelope.p,q,envelope.distances),
        })
        checks["transport_budget_violation"]=max(0.,checks["transport_cost"]-envelope.rho)
        checks["wasserstein_budget_violation"]=max(0.,checks["wasserstein_distance"]-envelope.rho)
    else:
        checks.update({"nominal_law_error":float(np.max(np.abs(q-envelope.p))),
                       "transport_budget_violation":0.,"wasserstein_budget_violation":0.})
    requirements={
        "demand_error":TOL.flow_feasibility,"nonnegative_flow_error":TOL.flow_feasibility,
        "probability_sum_error":TOL.probability,"probability_nonnegativity_error":TOL.probability,
        "tail_capacity_violation":TOL.probability,"tilt_identity_error":TOL.probability,
        "tail_value_error_per_demand":TOL.risk_per_demand,
        "inner_risk_gap_per_demand":TOL.risk_per_demand,
        "raw_law_risk_gap_per_demand":TOL.risk_per_demand,
        "kkt_stationarity":TOL.kkt,"pi_kernel_error":TOL.kkt,"pi_lp_error":TOL.kkt,
        "kernel_relative_error":TOL.kernel_relative,
        "global_gap_per_demand":TOL.global_gap_per_demand,
        "master_objective_error_per_demand":TOL.global_gap_per_demand,
        "master_feasibility_error":2e-7,
        "transport_budget_violation":TOL.probability,
        "wasserstein_budget_violation":TOL.probability,
    }
    if envelope.robust:
        requirements.update({"transport_row_error":TOL.probability,
                              "transport_column_error":TOL.probability,
                              "transport_nonnegativity_error":TOL.probability})
    else:requirements["nominal_law_error"]=TOL.probability
    failures={key:{"value":checks[key],"tolerance":threshold}
              for key,threshold in requirements.items()
              if not np.isfinite(checks[key]) or checks[key]>threshold}
    if UB<LB-1e-9:failures["reversed_bounds"]={"LB":LB,"UB":UB}
    checks["passed"]=not bool(failures)
    return c,pi,checks,failures



### Convex finite-scenario master programs with analytic Jacobians

In [5]:
# Convex finite-scenario master programs with analytic Jacobians


def polish_master_kkt(z0,obj,eq,je,ineq,ji,bounds):
    """Refine a candidate using its active-set KKT equations.

    This is only a numerical refinement, not the acceptance criterion. An
    independent envelope LP and global-gap certificate still decide acceptance.
    No optimizer-specific multiplier attribute is needed.
    """
    from scipy.optimize import lsq_linear
    z0=np.asarray(z0,dtype=float)
    d=len(z0)
    lower=np.array([-np.inf if a is None else a for a,b in bounds],dtype=float)
    upper=np.array([np.inf if b is None else b for a,b in bounds],dtype=float)
    initial_slack=ineq(z0)
    active_i=np.flatnonzero(initial_slack < 2e-7)
    active_lo=np.flatnonzero(np.isfinite(lower)&(z0-lower<2e-8))
    active_up=np.flatnonzero(np.isfinite(upper)&(upper-z0<2e-8))
    def slacks(z):
        return np.r_[ineq(z)[active_i],z[active_lo]-lower[active_lo],upper[active_up]-z[active_up]]
    def jac_slacks(z):
        return np.vstack([ji(z)[active_i],np.eye(d)[active_lo],-np.eye(d)[active_up]])
    J=jac_slacks(z0)
    design=np.column_stack([je(z0),J.T])
    grad=obj(z0)[1]
    dual0=lsq_linear(design,grad,bounds=(np.r_[-np.inf,np.zeros(len(J))],np.full(len(J)+1,np.inf)),
                      tol=1e-12,max_iter=300).x
    def residual(v):
        z=v[:d];pi=v[d];dual=v[d+1:]
        stationarity=obj(z)[1]-je(z)*pi-jac_slacks(z).T@dual
        return np.r_[stationarity,eq(z),slacks(z)]
    x0=np.r_[z0,dual0]
    lows=np.r_[lower,-np.inf,np.zeros(len(J))]
    highs=np.r_[upper,np.inf,np.full(len(J),np.inf)]
    # Start at bound values without modifying the candidate stored by the master.
    x0=np.minimum(np.maximum(x0,lows),highs)
    rr=least_squares(residual,x0,bounds=(lows,highs),jac='3-point',
                      ftol=1e-13,xtol=1e-13,gtol=1e-13,max_nfev=150,x_scale='jac')
    z=rr.x[:d]
    violation=max(abs(eq(z)),max(0.,-float(np.min(ineq(z)))) if len(ineq(z)) else 0.)
    # A complete convex KKT solution on a feasible point is safe to keep.
    if violation<=2e-9 and np.max(np.abs(residual(rr.x)))<=2e-8 and obj(z)[0]<=obj(z0)[0]+1e-8:
        return z,True,int(rr.nfev)
    return z0,False,int(rr.nfev)


def solve_convex_master(evaluator,center,rho,distances,initial_shares=None,ftol=1e-12):
    ev=evaluator;cfg=ev.cfg;n=ev.paths.n_paths;m=len(center)
    alpha,eta=cfg.alpha,cfg.eta;beta=1-alpha
    y0=np.full(n,1/n) if initial_shares is None else np.asarray(initial_shares,dtype=float).copy()
    if y0.shape!=(n,) or np.min(y0)<0 or abs(y0.sum()-1)>1e-10:
        raise ValueError("Invalid convex-master initial shares.")
    l0,T0,_,_=ev.values(y0)
    nominal=(rho is None or rho==0)
    if nominal and eta==0:
        def obj(z):
            l,T,_,_=ev.values(z);H,dH=ev.regularizer(z)
            return float(center@l+H),center@T+dH
        z0=y0; bounds=[(0.,1.)]*n
        def ineq(z):return np.empty(0)
        def ji(z):return np.empty((0,n))
        it=None;ik=None
    elif nominal:
        it=n
        z0=np.r_[y0,float(center@l0),np.maximum(l0-center@l0,0)]
        bounds=[(0.,1.)]*n+[(None,None)]+[(0.,None)]*m
        def obj(z):
            l,T,_,_=ev.values(z[:n]);H,dH=ev.regularizer(z[:n])
            value=(1-eta)*(center@l)+eta*z[n]+eta/beta*(center@z[n+1:])+H
            grad=np.r_[(1-eta)*(center@T)+dH,eta,eta/beta*center]
            return float(value),grad
        def ineq(z):
            l,_,_,_=ev.values(z[:n]);return z[n+1:]+z[n]-l
        def ji(z):
            _,T,_,_=ev.values(z[:n]);return np.column_stack([-T,np.ones(m),np.eye(m)])
        ik=None
    else:
        observed=np.flatnonzero(center>0);weights=center[observed];o=len(observed)
        D=np.asarray(distances)[observed,:]
        k0=1.
        if eta==0:
            ik=n;it=None
            z0=np.r_[y0,k0,np.max(l0[None,:]-k0*D,axis=1)]
            bounds=[(0.,1.)]*n+[(0.,None)]+[(None,None)]*o
            def obj(z):
                H,dH=ev.regularizer(z[:n])
                return float(rho*z[n]+weights@z[n+1:]+H),np.r_[dH,rho,weights]
            def ineq(z):
                l,_,_,_=ev.values(z[:n]);return (z[n+1:,None]+z[n]*D-l[None,:]).ravel()
            def ji(z):
                _,T,_,_=ev.values(z[:n]);return np.column_stack([np.tile(-T,(o,1)),D.ravel(),np.repeat(np.eye(o),m,axis=0)])
        else:
            it=n;ik=n+1;ss=slice(n+2,n+2+o);uu=slice(n+2+o,n+2+o+m)
            t0=float(center@l0);u0=np.maximum(l0-t0,0)
            g0=(1-eta)*l0+eta/beta*u0
            z0=np.r_[y0,t0,k0,np.max(g0[None,:]-k0*D,axis=1),u0]
            bounds=[(0.,1.)]*n+[(None,None),(0.,None)]+[(None,None)]*o+[(0.,None)]*m
            def obj(z):
                H,dH=ev.regularizer(z[:n])
                return float(eta*z[it]+rho*z[ik]+weights@z[ss]+H),np.r_[dH,eta,rho,weights,np.zeros(m)]
            def ineq(z):
                l,_,_,_=ev.values(z[:n]);u=z[uu]
                g=(1-eta)*l+eta/beta*u
                return np.r_[u+z[it]-l,(z[ss,None]+z[ik]*D-g[None,:]).ravel()]
            def ji(z):
                _,T,_,_=ev.values(z[:n])
                J=np.zeros((m+o*m,len(z)))
                J[:m,:n]=-T;J[:m,it]=1;J[:m,uu]=np.eye(m)
                J[m:,:n]=np.tile(-(1-eta)*T,(o,1))
                J[m:,ik]=D.ravel();J[m:,ss]=np.repeat(np.eye(o),m,axis=0)
                J[m:,uu]=np.tile(-eta/beta*np.eye(m),(o,1))
                return J
    def eq(z):return float(np.sum(z[:n])-1)
    def je(z):return np.r_[np.ones(n),np.zeros(len(z)-n)]
    constraints=[{"type":"eq","fun":eq,"jac":je}]
    if len(ineq(z0)):
        constraints.append({"type":"ineq","fun":ineq,"jac":ji})
    res=minimize(obj,z0,jac=True,method="SLSQP",bounds=bounds,constraints=constraints,
                 options={"ftol":ftol,"maxiter":3000,"disp":False})
    refined,polished,polish_nfev=polish_master_kkt(res.x,obj,eq,je,ineq,ji,bounds)
    res.x=refined
    res.fun=obj(refined)[0]
    res["polished"]=polished
    res["polish_nfev"]=polish_nfev
    residual=max(abs(eq(res.x)),max(0.,-float(np.min(ineq(res.x)))) if len(ineq(res.x)) else 0.)
    # No post-solve change in flow. Demand feasibility is checked on this vector.
    return res,res.x[:n].copy(),float(res.x[it]) if it is not None else None,\
        float(res.x[ik]) if ik is not None else None,float(residual)


def solve_model_b(instance,paths,cfg,center=None,rho=None,distances=None,initial_flows=None):
    """Convex Model B/SP/DRO solve with a joint risk-and-equilibrium certificate."""
    ev=CostEvaluator(instance,paths,cfg)
    p=checked_probabilities(instance.probabilities if center is None else center,len(instance.scenario_names))
    env=FiniteRiskEnvelope(p,cfg.alpha,cfg.eta,rho,distances)
    start=time.perf_counter()
    initial=None if initial_flows is None else np.asarray(initial_flows,dtype=float)/cfg.q
    attempts=[]
    for attempt in range(4):
        res,y,gamma,kappa,feasibility=solve_convex_master(ev,p,rho,distances,initial,ftol=[1e-11,1e-10,1e-12,1e-10][attempt])
        l,T,_,_=ev.values(y);H,dH=ev.regularizer(y)
        selection=select_kkt_law(env,l,T,dH,y)
        c,pi,checks,failures=certify_model_b(ev,y,env,selection,float(res.fun),feasibility)
        checks["solver_success"]=bool(res.success)
        checks["solver_status"]=str(res.message)
        checks["solver_attempts"]=attempt+1
        checks["master_polished"]=bool(res.polished)
        checks["polish_nfev"]=int(res.polish_nfev)
        if not failures and res.success:
            cfg_use=cfg
            result=SolvedEquilibrium(
                "Model B-DRO" if env.robust else "Model B",instance.name,cfg_use,cfg.q*y,c,pi,
                selection["q"],selection["tail"],selection["tilt"],selection["plan"],
                cfg.q*(selection["risk_max"]+H),cfg.q*selection["risk_max"],
                None if gamma is None else cfg.q*gamma,
                None if kappa is None else cfg.q*kappa,int(res.nit),
                time.perf_counter()-start,checks,tail_law_identified=(cfg.eta>0))
            return result
        attempts.append({"attempt":attempt+1,"solver":res.message,"failures":failures})
        if attempt==0 and np.min(y)>=0 and abs(y.sum()-1)<1e-10:
            initial=y
        else:
            # Deterministic, feasible alternative starts; no randomness or hidden seeds.
            initial=np.full(paths.n_paths,0.15/(paths.n_paths-1))
            initial[attempt%paths.n_paths]=0.85
    raise NumericalValidationError(f"No certified Model B result for {instance.name}, alpha={cfg.alpha}, eta={cfg.eta}, rho={rho}: {attempts}")



### Manuscript Table 2 export and synchronization checks

In [6]:
# Table 2 publication export: one solved source, no stored numerical table.
# This cell does not modify the equilibrium model or the solver tolerances.
import os
import re
import tempfile

TABLE2_EXPORT_VERSION = "table2-single-source-sync-v2"
TABLE2_LABEL = "tab:braess_modelB_direct_eta_grid_full"
TABLE2_FILENAME = "braess_modelB_direct_eta_table_full.tex"
TABLE2_SPECS = [
    ("RiskCase", "Risk case", None),
    ("alpha", r"$\alpha$", ".2f"),
    ("eta", r"$\eta$", ".1f"),
    *[(f"f{k}_{model}_over_q", rf"$f_{k}^{model}/q$", ".3f")
      for model in ("A", "B") for k in (1, 2, 3)],
    ("pi_A", r"$\pi^A$", ".2f"),
    ("pi_B", r"$\pi^B$", ".2f"),
    ("D_tail_AB", r"$D_{\rm tail}^{AB}$", ".3f"),
    ("D_p_AB", r"$D_p^{AB}$", ".3f"),
    ("D_f_AB", r"$D_f^{AB}$", "small3"),
    ("Gap_A_at_fB", r"$\mathrm{Gap}_A(f^B)$", "small4"),
    ("Active_A_minus_Active_B", r"$|\mathcal A^A|-|\mathcal A^B|$", ".0f"),
]


def _table2_key(alpha, eta):
    return (round(float(alpha), 12), round(float(eta), 12))


def _table2_check(condition, message):
    if not bool(condition):
        raise NumericalValidationError(message)


def _table2_format(value, fmt):
    if fmt is None:
        return str(value).replace("Mean-CVaR", "Mean--CVaR")
    value = float(value)
    _table2_check(np.isfinite(value), "A nonfinite value cannot enter Table 2.")
    if fmt.startswith("small"):
        decimals = int(fmt[-1])
        _table2_check(value >= 0, "A distance or VI gap cannot be negative.")
        if value == 0:
            return "0"
        if value < 10.0 ** (-decimals):
            mantissa, exponent = format(value, ".1e").split("e")
            return rf"${mantissa}\times10^{{{int(exponent)}}}$"
        return format(value, f".{decimals}f")
    text = format(value, fmt)
    return text[1:] if text.startswith("-") and float(text) == 0 else text


def _table2_expected_cells(row):
    return [_table2_format(row[key], fmt) for key, _, fmt in TABLE2_SPECS]


def parse_table2_rows(text):
    """Read only 16-column data rows; also accept an earlier table for comparison."""
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line.startswith(("Expectation-only &", "Mean--CVaR &", "Mean-CVaR &", "CVaR-only &")):
            continue
        _table2_check(line.endswith(r"\\"), "A Table 2 data row has no LaTeX row terminator.")
        cells = [part.strip() for part in line[:-2].split("&")]
        _table2_check(len(cells) == len(TABLE2_SPECS), "A Table 2 data row does not have 16 columns.")
        rows.append(cells)
    _table2_check(bool(rows), "No Table 2 data rows were found.")
    keys = [_table2_key(cells[1], cells[2]) for cells in rows]
    _table2_check(len(set(keys)) == len(keys), "Duplicate (alpha, eta) rows in Table 2.")
    return rows


def _table2_numeric_token(token):
    """Return the displayed number and its last-place quantum, independently."""
    clean = token.strip().strip("$")
    scientific = re.fullmatch(r"([+-]?\d+(?:\.\d+)?)\\times10\^\{([+-]?\d+)\}", clean)
    if scientific:
        mantissa, exp = scientific.groups()
        places = len(mantissa.partition(".")[2])
        return float(mantissa) * 10.0 ** int(exp), 10.0 ** (int(exp) - places)
    value = float(clean)
    if "e" in clean.lower():
        mantissa, exp = clean.lower().split("e")
        return value, 10.0 ** (int(exp) - len(mantissa.partition(".")[2]))
    return value, 10.0 ** (-len(clean.partition(".")[2]))


def validate_table2_source(df, all_results, laws, alphas, etas):
    """Check every published diagnostic against the very same solved records."""
    expected_keys = [_table2_key(a, e) for a, e in itertools.product(alphas, etas)]
    actual_keys = [_table2_key(row.alpha, row.eta) for row in df.itertuples()]
    _table2_check(actual_keys == expected_keys, "The table grid or row order is incorrect.")
    _table2_check(len(set(actual_keys)) == len(actual_keys), "The requested parameter grid has duplicates.")
    _table2_check(len(all_results) == 2 * len(df), "Expected one independently solved A and B record per case.")
    _table2_check(all_results["passed"].eq(True).all(), "An uncertified result cannot enter Table 2.")
    numeric = [key for key, _, fmt in TABLE2_SPECS if fmt is not None]
    _table2_check(np.isfinite(df[numeric].to_numpy(dtype=float)).all(), "Nonfinite Table 2 values.")
    inst = build_route_local_instance()
    paths = base_path_set()
    compare_count = 0
    max_error = 0.0

    def compare(actual, expected, name, tolerance=2e-11):
        nonlocal compare_count, max_error
        error = float(np.max(np.abs(np.asarray(actual, float) - np.asarray(expected, float))))
        _table2_check(error <= tolerance, f"Source mismatch in {name}: {error:.6g}.")
        max_error = max(max_error, error)
        compare_count += int(np.asarray(actual).size)

    for _, row in df.iterrows():
        key = _table2_key(row.alpha, row.eta)
        matching = all_results[[ _table2_key(a, e) == key
                                for a, e in zip(all_results.alpha, all_results.eta) ]]
        _table2_check(set(matching.Model) == {"Model A", "Model B"}, "Incorrect model records for a table row.")
        a = matching[matching.Model == "Model A"].iloc[0]
        b = matching[matching.Model == "Model B"].iloc[0]
        cfg = replace(SolverConfig(), alpha=float(row.alpha), eta=float(row.eta))
        ev = CostEvaluator(inst, paths, cfg)
        fa = np.array([a[f"f{k}"] for k in (1, 2, 3)], float)
        fb = np.array([b[f"f{k}"] for k in (1, 2, 3)], float)
        cb = np.array([b[f"Cost{k}"] for k in (1, 2, 3)], float)
        tail = np.array([b[f"PureTail{s}"] for s in SCENARIO_KEYS], float)
        tilt = np.array([b[f"RobustTilt{s}"] for s in SCENARIO_KEYS], float)
        pure_rows = laws[[ _table2_key(a_, e_) == key for a_, e_ in zip(laws.alpha, laws.eta) ]]
        _table2_check(list(pure_rows.Scenario) == list(inst.scenario_names), "Scenario order or coverage differs.")
        compare(pure_rows.SelectedPureTail_B.to_numpy(), tail, "selected Model B pure tail")
        compare(pure_rows.SelectedTilt_B.to_numpy(), tilt, "selected Model B tilt")
        compare(tilt, (1-cfg.eta)*inst.probabilities + cfg.eta*tail, "tilt identity")
        _, path_times, _, _ = ev.values(fb/cfg.q)
        compare(cb, path_times.T @ tilt, "Model B costs under selected law")
        F = cb + np.log1p(fb/cfg.r0)/cfg.theta
        active_b = fb > 1e-6
        active_a = fa > 1e-6
        compare(row.pi_B, np.mean(F[active_b]), "pi_B from active-path KKT costs")
        _table2_check(np.max(np.abs(F[active_b]-row.pi_B)) <= TOL.kkt,
                      "Displayed pi_B is not KKT compatible.")
        if np.any(~active_b):
            _table2_check(np.min(F[~active_b]) >= row.pi_B-TOL.kkt, "Inactive-path KKT violation.")
        _, tails_a, tilts_a = pathwise_costs(ev, fb/cfg.q)
        dt = float(np.max(np.sum(np.abs(tails_a-tail[None, :]), axis=1)))
        dp = float(np.max(np.sum(np.abs(tilts_a-tilt[None, :]), axis=1)))
        _, _, diag_a_at_b = model_a_diagnostics(ev, fb/cfg.q)
        values = {
            "pi_A": a.pi_od, "pi_B": b.pi_od,
            "D_tail_AB": dt, "D_p_AB": dp,
            "D_f_AB": np.sum(np.abs(fa-fb))/cfg.q,
            "Gap_A_at_fB": diag_a_at_b["vi_gap_relative"],
            "Active_A_minus_Active_B": int(active_a.sum()-active_b.sum()),
        }
        for k in (1, 2, 3):
            values[f"f{k}_A_over_q"] = fa[k-1]/cfg.q
            values[f"f{k}_B_over_q"] = fb[k-1]/cfg.q
        for column, expected in values.items():
            compare(row[column], expected, column)
        compare(row.D_p_AB, row.eta*row.D_tail_AB, "D_p = eta D_tail", tolerance=1e-9)
        _table2_check(row.RiskCase == risk_case(cfg.eta), "Incorrect risk-case label.")
    return {"passed": True, "parameter_cases": len(df),
            "scalar_comparisons": compare_count, "maximum_source_difference": max_error}


def render_table2_manuscript(df):
    """Keep the uploaded table's label, 16 columns, grouping, and display precision."""
    alpha_text = ",".join(f"{v:.2f}" for v in df.alpha.unique())
    eta_text = ",".join("0" if v == 0 else f"{v:.1f}" for v in df.eta.unique())
    caption = (
        r"Direct-$(\alpha,\eta)$ grid with Model A--Model B flow, reservation-cost, tail-law, and VI diagnostics, "
        r"reported on the retained grid $\alpha\in\{" + alpha_text + r"\}$ and $\eta\in\{" + eta_text + r"\}$. "
        r"The table reports normalized path flows, $f_i^A/q$ and $f_i^B/q$, and endogenous reservation "
        r"generalized costs $\pi^A$ and $\pi^B$. Small nonzero $D_f^{AB}$ and $\mathrm{Gap}_A(f^B)$ "
        r"values are printed in scientific notation rather than rounded to zero. "
        r"$D_{\rm tail}^{AB}$ is the maximum pathwise $\ell_1$ distance between pure CVaR-tail laws "
        r"$p_s\chi_s/(1-\alpha)$; $D_p^{AB}$ is the corresponding distance between the effective "
        r"tail-tilted laws in \eqref{eq:tail_tilt_prob}. Both law distances are evaluated at $f^B$, "
        r"using the same KKT-compatible Model B selector used to compute $\pi^B$. "
        r"At $\eta=0$, the pure-tail diagnostic is a value-only reporting convention and does not affect "
        r"the equilibrium. Displayed flow shares may not sum to one because of rounding."
    )
    lines = [
        "% Generated from this run's table_2_SOURCE_FULL.csv; do not copy numbers from an older table.",
        "% Model B laws, costs, and pi_B are selected jointly with equilibrium KKT conditions.",
        "% Required manuscript packages: amsmath, booktabs, graphicx.",
        r"\begin{table}[htbp]", r"\centering", r"\tiny",
        r"\setlength{\tabcolsep}{1.45pt}", r"\renewcommand{\arraystretch}{1.05}",
        "\\caption{" + caption + "}", "\\label{" + TABLE2_LABEL + "}",
        r"\resizebox{\textwidth}{!}{%", r"\begin{tabular}{lrrrrrrrrrrrrrrr}",
        r"\toprule", " & ".join(header for _, header, _ in TABLE2_SPECS) + r"\\", r"\midrule",
    ]
    previous_alpha = None
    for _, row in df.iterrows():
        if previous_alpha is not None and row.alpha != previous_alpha:
            lines.append(r"\addlinespace[0.25em]")
        lines.append(" & ".join(_table2_expected_cells(row)) + r"\\")
        previous_alpha = row.alpha
    lines += [r"\bottomrule", r"\end{tabular}%", "}", r"\end{table}"]
    return "\n".join(lines) + "\n"


def validate_table2_exports(df, tex_path, csv_path):
    """Re-read the real exported bytes, not an in-memory copy, and audit all cells."""
    exported = pd.read_csv(csv_path, float_precision="round_trip")
    columns = [key for key, _, _ in TABLE2_SPECS]
    _table2_check(list(exported.columns) == list(df.columns), "Source CSV column schema differs.")
    _table2_check(len(exported) == len(df), "Source CSV row count differs.")
    for key, _, fmt in TABLE2_SPECS:
        if fmt is None:
            _table2_check(list(exported[key]) == list(df[key]), f"CSV text mismatch: {key}.")
        else:
            _table2_check(np.array_equal(exported[key].to_numpy(float), df[key].to_numpy(float)),
                          f"CSV did not round-trip exactly for {key}.")
    text = Path(tex_path).read_text(encoding="utf-8")
    _table2_check(text.count("\\label{"+TABLE2_LABEL+"}") == 1, "Table 2 label changed or is duplicated.")
    parsed = parse_table2_rows(text)
    _table2_check(len(parsed) == len(df), "The LaTeX table has missing or extra rows.")
    _table2_check(text.count(r"\addlinespace[0.25em]") == df.alpha.nunique()-1,
                  "Incorrect alpha-group spacing in LaTeX table.")
    numeric_count = 0
    max_rounding_units = 0.0
    for index, (_, row) in enumerate(df.iterrows()):
        expected = _table2_expected_cells(row)
        _table2_check(parsed[index] == expected, f"LaTeX row {index+1} differs from the solved data.")
        for j, (key, _, fmt) in enumerate(TABLE2_SPECS):
            if fmt is None:
                continue
            number, quantum = _table2_numeric_token(parsed[index][j])
            original = float(row[key])
            delta = abs(number-original)
            allowed = 0.5*quantum + 1e-13*max(abs(number), abs(original), 1e-100)
            _table2_check(delta <= allowed, f"Printed rounding error is too large in {key}, row {index+1}.")
            if fmt.startswith("small") and original != 0:
                _table2_check(number != 0, f"A small nonzero {key} was printed as zero.")
            max_rounding_units = max(max_rounding_units, delta/quantum)
            numeric_count += 1
    return {"passed": True, "rows_checked": len(df), "numeric_cells_checked": numeric_count,
            "text_cells_checked": len(df), "csv_numeric_round_trip_exact": True,
            "all_tex_cells_match_formatted_source": True,
            "maximum_rounding_error_in_last_place_units": max_rounding_units,
            "tex_sha256": hashlib.sha256(Path(tex_path).read_bytes()).hexdigest(),
            "csv_sha256": hashlib.sha256(Path(csv_path).read_bytes()).hexdigest()}


def write_table2_publication_bundle(df, all_results, laws, outdir, alphas, etas):
    outdir = Path(outdir)
    source_report = validate_table2_source(df, all_results, laws, alphas, etas)
    tex_path = outdir / TABLE2_FILENAME
    csv_path = outdir / "table_2_SOURCE_FULL.csv"
    # run_experiment_01 has just written this CSV from df, in full code precision.
    tex_path.write_text(render_table2_manuscript(df), encoding="utf-8")
    export_report = validate_table2_exports(df, tex_path, csv_path)
    report = {"export_version": TABLE2_EXPORT_VERSION, "all_checks_passed": True,
              "source_validation": source_report, "export_validation": export_report,
              "scope": "All rows and columns in the provided direct-alpha-eta grid; floating-point numerical checks."}
    (outdir/"TABLE2_EXPORT_VALIDATION.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


def compare_table2_texts(old_text, new_text):
    """Audit only: old cells are never used to compute or fill the new table."""
    old_rows = {_table2_key(c[1], c[2]): c for c in parse_table2_rows(old_text)}
    records = []
    for new in parse_table2_rows(new_text):
        key = _table2_key(new[1], new[2])
        old = old_rows.get(key)
        for j, (column, _, _) in enumerate(TABLE2_SPECS):
            if old is not None and old[j] == new[j]:
                continue
            records.append({"alpha": key[0], "eta": key[1], "column": column,
                            "previous_display": None if old is None else old[j],
                            "updated_display": new[j]})
    return pd.DataFrame(records, columns=["alpha", "eta", "column", "previous_display", "updated_display"])


def publish_table2_outputs(result, target):
    """Update the explicitly configured manuscript table, after all checks pass.

    The previous bytes are backed up. An unrelated file or concurrent edit is
    never silently overwritten. Data and labels come only from this run.
    """
    target = Path(target).expanduser().resolve()
    _table2_check(target.suffix.lower() == ".tex", "PAPER_TABLE_PATH must name a .tex file.")
    outdir = Path(result["output_dir"]).resolve()
    generated = outdir/TABLE2_FILENAME
    csv_path = outdir/"table_2_SOURCE_FULL.csv"
    validation = validate_table2_exports(result["table"], generated, csv_path)
    audit_dir = outdir/"publication_audit"
    audit_dir.mkdir(parents=True, exist_ok=True)
    payload = generated.read_bytes()
    previous = target.read_bytes() if target.exists() else None
    backup_path = None
    if previous is not None and target != generated:
        old_text = previous.decode("utf-8-sig")
        stripped = "\n".join(line for line in old_text.splitlines() if not line.lstrip().startswith("%")).strip()
        _table2_check(stripped.startswith(r"\begin{table}") and stripped.endswith(r"\end{table}"),
                      "The publication target is not a standalone table; it was not overwritten.")
        _table2_check(old_text.count(r"\begin{table}") == 1 and
                      old_text.count("\\label{"+TABLE2_LABEL+"}") == 1,
                      "The publication target has a different or duplicate label; it was not overwritten.")
        parse_table2_rows(old_text)
        if previous != payload:
            backup_path = audit_dir/"previous_table_before_update.tex.bak"
            # Never destroy an earlier backup if this publication cell is re-run.
            if backup_path.exists():
                backup_path = audit_dir/("previous_table_"+hashlib.sha256(previous).hexdigest()[:16]+".tex.bak")
            backup_path.write_bytes(previous)
            changes = compare_table2_texts(old_text, payload.decode("utf-8"))
            changes.to_csv(audit_dir/"table2_changed_cells.csv", index=False)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target != generated:
        temp_path = None
        try:
            with tempfile.NamedTemporaryFile(mode="wb", dir=target.parent, prefix=".table2_", suffix=".tmp", delete=False) as tmp:
                temp_path = Path(tmp.name)
                tmp.write(payload)
            _table2_check(temp_path.read_bytes() == payload, "Temporary publication copy differs.")
            current = target.read_bytes() if target.exists() else None
            _table2_check(current == previous, "The paper table changed during export; it was not overwritten.")
            os.replace(temp_path, target)
        finally:
            if temp_path is not None and temp_path.exists():
                temp_path.unlink()
    _table2_check(target.read_bytes() == payload, "Publication copy differs from the checked LaTeX file.")
    final_check = validate_table2_exports(result["table"], target, csv_path)
    report = {"all_checks_passed": True, "export_version": TABLE2_EXPORT_VERSION,
              "publication_target": str(target), "checked_run_table": str(generated),
              "source_csv": str(csv_path), "previous_table_backup": str(backup_path) if backup_path else None,
              "publication_matches_checked_run_byte_for_byte": True,
              **final_check}
    (outdir/"TABLE2_PUBLICATION_REPORT.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return {"target": target, "csv_path": csv_path, "report": report, "backup": backup_path}


def run_table2_export_regressions(result):
    """Deliberately corrupt copies and ensure no stale or unrelated output is accepted."""
    outdir = Path(result["output_dir"])
    df = result["table"]
    tex_path = outdir/TABLE2_FILENAME
    csv_path = outdir/"table_2_SOURCE_FULL.csv"
    laws = pd.read_csv(outdir/"table_2_selected_modelB_laws_FULL.csv", float_precision="round_trip")
    checks = []

    def must_reject(name, operation):
        try:
            operation()
        except NumericalValidationError:
            checks.append({"Test": name, "Passed": True})
        else:
            raise NumericalValidationError("A corrupted export was accepted: " + name)

    with tempfile.TemporaryDirectory(prefix="table2_export_checks_") as folder:
        folder = Path(folder)
        bad_table = df.copy(deep=True)
        bad_table.loc[bad_table.index[0], "pi_B"] += 1.0
        must_reject("Reject a changed pi_B before export", lambda: validate_table2_source(
            bad_table, result["all_results"], laws, df.alpha.unique(), df.eta.unique()))

        text = tex_path.read_text(encoding="utf-8")
        first = parse_table2_rows(text)[0]
        corrupt = first.copy()
        corrupt[10] = format(float(first[10])+1, ".2f")
        bad_tex = folder/"bad_table.tex"
        bad_tex.write_text(text.replace(" & ".join(first)+r"\\", " & ".join(corrupt)+r"\\", 1), encoding="utf-8")
        must_reject("Reject a stale LaTeX pi_B cell", lambda: validate_table2_exports(df, bad_tex, csv_path))

        bad_csv = folder/"bad_source.csv"
        dataframe_to_csv(bad_table, bad_csv)
        must_reject("Reject CSV/solved-source mismatch", lambda: validate_table2_exports(df, tex_path, bad_csv))

        unrelated = folder/"unrelated_manuscript.tex"
        sentinel = r"\documentclass{article}\begin{document}Unrelated file.\end{document}"
        unrelated.write_text(sentinel, encoding="utf-8")
        must_reject("Refuse to overwrite an unrelated LaTeX document", lambda: publish_table2_outputs(result, unrelated))
        _table2_check(unrelated.read_text(encoding="utf-8") == sentinel, "The unrelated file was changed.")
    report = pd.DataFrame(checks)
    report.to_csv(outdir/"TABLE2_EXPORT_REGRESSION_TESTS.csv", index=False)
    return report


### Shared experiment diagnostics and transparent exports

In [7]:
# Shared experiment diagnostics and transparent exports

SCENARIO_KEYS=("Normal","Upper","Lower","Shortcut","SystemWide")
TABLE_ALPHAS=[0.80,0.84,0.88,0.92,0.96]
TABLE_ETAS=[0.0,0.2,0.4,0.6,0.8,1.0]


def risk_case(eta):
    return "Expectation-only" if eta==0 else ("CVaR-only" if eta==1 else "Mean-CVaR")


def result_record(result,rho=0.,network="Before"):
    row={"RiskCase":risk_case(result.cfg.eta),"alpha":result.cfg.alpha,"eta":result.cfg.eta,
         "rho":float(rho),"Model":result.model,"Instance":result.instance,"Network":network,
         "q":result.cfg.q,"theta":result.cfg.theta,"r0":result.cfg.r0,
         "pi_od":result.pi,"ActiveCount":result.active_count,
         "Objective":result.objective,"RiskValue":result.risk_value,
         "t_RU":result.gamma,"kappa":result.kappa,
         "Iterations":result.iterations,"ElapsedSeconds":result.elapsed_seconds,
         "TailIdentifiedByRiskObjective":result.tail_law_identified}
    for k,(f,p,c) in enumerate(zip(result.flows,result.shares,result.path_costs),start=1):
        row[f"f{k}"]=float(f);row[f"P{k}"]=float(p);row[f"Cost{k}"]=float(c)
    for prefix,vector in [("QStar",result.raw_law),("PureTail",result.tail_law),("RobustTilt",result.tilted_law)]:
        if vector is not None:
            for key,value in zip(SCENARIO_KEYS,vector):row[prefix+key]=float(value)
    row.update(result.checks)
    return row


def ab_metrics(instance,paths,cfg,a,b):
    ev=CostEvaluator(instance,paths,cfg)
    _,tail_a,tilt_a=pathwise_costs(ev,b.shares)
    d_tail=np.sum(np.abs(tail_a-b.tail_law[None,:]),axis=1)
    d_p=np.sum(np.abs(tilt_a-b.tilted_law[None,:]),axis=1)
    _,_,a_at_b=model_a_diagnostics(ev,b.shares)
    active_a=a.flows>1e-6;active_b=b.flows>1e-6
    row={"RiskCase":risk_case(cfg.eta),"alpha":cfg.alpha,"eta":cfg.eta,
         "pi_A":a.pi,"pi_B":b.pi,"D_tail_AB":float(d_tail.max()),
         "D_p_AB":float(d_p.max()),"D_f_AB":float(np.linalg.norm(a.flows-b.flows,ord=1)/cfg.q),
         "Gap_A_at_fB":float(a_at_b['vi_gap_relative']),
         "Gap_A_at_fA":float(a.checks['vi_gap_relative']),
         "Active_A_minus_Active_B":int(active_a.sum()-active_b.sum()),
         "ActiveSetSymmetricDifference":int(np.count_nonzero(active_a!=active_b)),
         "Active_A":int(active_a.sum()),"Active_B":int(active_b.sum()),
         "KKT_B":float(b.checks['kkt_stationarity']),
         "GlobalGap_B":float(b.checks['upper_bound']-b.checks['lower_bound']),
         "InnerRiskGap_B_per_demand":float(b.checks['inner_risk_gap_per_demand']),
         "TailLawAtEtaZeroIsValueOnly":bool(cfg.eta==0)}
    for k in range(paths.n_paths):
        row[f"f{k+1}_A_over_q"]=float(a.shares[k]);row[f"f{k+1}_B_over_q"]=float(b.shares[k])
        row[f"P{k+1}_A"]=float(a.shares[k]);row[f"P{k+1}_B"]=float(b.shares[k])
        row[f"D_tail_P{k+1}"]=float(d_tail[k]);row[f"D_p_P{k+1}"]=float(d_p[k])
    if abs(row['D_p_AB']-cfg.eta*row['D_tail_AB'])>1e-9:
        raise NumericalValidationError("D_p must equal eta*D_tail at a common evaluation flow.")
    return row


def evaluate_test(instance,paths,result,probabilities=TEST_LAW):
    ev=CostEvaluator(instance,paths,result.cfg)
    ell,T,times,x=ev.values(result.shares)
    L=result.cfg.q*ell
    p=checked_probabilities(probabilities,len(L));a=result.cfg.alpha;e=result.cfg.eta
    mean=float(p@L);cv=ru_cvar_independent(L,p,a)
    tstt=times@x
    avg=tstt/result.cfg.q
    return {"MeanTest":mean,"CVaRTest":cv,"CETest":(1-e)*mean+e*cv,
            "MeanTravelTimeTest":float(p@avg),
            "CVaRMeanTravelTimeTest":ru_cvar_independent(avg,p,a),
            "MeanTSTTTest":float(p@tstt)}


def dataframe_to_csv(df,path):
    path=Path(path)
    if len(df)==0:raise NumericalValidationError("Refusing to export an empty data table.")
    df.to_csv(path,index=False,float_format="%.17g")


def write_latex_table(df,path,spec,caption,label):
    """spec entries: (source-column, TeX-header, Python format or None)."""
    lines=[r"\begin{table}[htbp]",r"\centering",r"\scriptsize",
           r"\setlength{\tabcolsep}{3pt}","\\caption{"+caption+"}","\\label{"+label+"}",
           r"\resizebox{\textwidth}{!}{%",r"\begin{tabular}{"+"l"+"r"*(len(spec)-1)+"}",
           r"\toprule"," & ".join(s[1] for s in spec)+r" \\",r"\midrule"]
    for _,row in df.iterrows():
        cells=[]
        for key,header,fmt in spec:
            value=row[key]
            if fmt is None:
                text=str(value).replace("Mean-CVaR","Mean--CVaR").replace("_",r"\_")
            else:
                if not np.isfinite(float(value)):
                    raise NumericalValidationError(f"Nonfinite published value {key}: {value}")
                text=format(float(value),fmt)
                if text.startswith("-") and float(text)==0:
                    text=text[1:]  # display-only removal of negative rounded zero
            cells.append(text)
        lines.append(" & ".join(cells)+r" \\")
    lines += [r"\bottomrule",r"\end{tabular}%",r"}",r"\end{table}"]
    Path(path).write_text("\n".join(lines)+"\n",encoding="utf-8")


def export_input_parameters(instance,paths,cfg,outdir):
    rows=[]
    for j,s in enumerate(instance.scenario_names):
        for link in LINKS:
            prm=instance.link_params[s][link]
            rows.append({"Scenario":s,"NominalProbability":instance.probabilities[j],"Link":link,
                         "t0":prm.t0,"capacity":prm.capacity,"delay":prm.delay,
                         "BPR_beta":cfg.bpr_beta,"BPR_gamma":cfg.bpr_gamma})
    dataframe_to_csv(pd.DataFrame(rows),Path(outdir)/"input_scenario_link_parameters_FULL.csv")
    pd.DataFrame(paths.incidence,index=paths.path_names,columns=LINKS).to_csv(Path(outdir)/"input_path_incidence.csv")


def write_validation_summary(records,outdir,extra=None):
    outdir=Path(outdir)
    df=pd.DataFrame(records)
    if not len(df) or not df['passed'].eq(True).all():
        raise NumericalValidationError("Some results did not pass validation.")
    dataframe_to_csv(df,outdir/"validation_details_FULL.csv")
    metrics=[c for c in df.columns if any(t in c for t in ['error','gap','violation','kkt_stationarity']) and pd.api.types.is_numeric_dtype(df[c])]
    summary={"core_version":CORE_VERSION,"all_checks_passed":True,"number_of_equilibrium_records":len(df),
             "maxima":{c:float(df[c].max()) for c in metrics if df[c].notna().any()},
             "tolerances":vars(TOL),"python":sys.version,"numpy":np.__version__,
             "scipy":scipy.__version__,"pandas":pd.__version__,"platform":platform.platform(),
             "certification_scope":"Finite support, fixed candidate paths, floating-point tolerances; not interval arithmetic or validation of empirical model assumptions."}
    if extra:summary.update(extra)
    (outdir/"VALIDATION_REPORT.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
    return summary


def _prepare_output(outdir):
    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)
    return outdir


def _save_figure(fig,path,dpi=600):
    import matplotlib.pyplot as plt
    fig.savefig(path,dpi=dpi,bbox_inches='tight')
    plt.close(fig)


def run_experiment_01(outdir,alphas=TABLE_ALPHAS,etas=TABLE_ETAS):
    """Table 2: original 30-case direct risk grid, corrected law and pi."""
    outdir=_prepare_output(outdir);cfg0=SolverConfig();paths=base_path_set();inst=build_route_local_instance()
    rows=[];full=[];law_rows=[]
    for alpha,eta in itertools.product(alphas,etas):
        cfg=replace(cfg0,alpha=float(alpha),eta=float(eta))
        b=solve_model_b(inst,paths,cfg)
        # Independently solve Model A rather than copying the Model B solution.
        a=solve_model_a(inst,paths,cfg)
        rows.append(ab_metrics(inst,paths,cfg,a,b))
        full.extend([result_record(a),result_record(b)])
        for j,scenario in enumerate(inst.scenario_names):
            law_rows.append({"alpha":alpha,"eta":eta,"Scenario":scenario,"Nominal":inst.probabilities[j],
                             "SelectedPureTail_B":b.tail_law[j],"SelectedTilt_B":b.tilted_law[j],
                             "TailIdentifiedByRiskObjective":b.tail_law_identified})
    df=pd.DataFrame(rows);all_df=pd.DataFrame(full)
    # Core identities checked again before any paper table is written.
    if not all_df['passed'].eq(True).all():raise NumericalValidationError("Table 2 contains an unvalidated solution.")
    dataframe_to_csv(all_df,outdir/"braess_direct_eta_all_models.csv")
    dataframe_to_csv(df,outdir/"braess_modelB_direct_eta_with_AB_flows_diagnostics.csv")
    dataframe_to_csv(df,outdir/"table_2_SOURCE_FULL.csv")
    dataframe_to_csv(pd.DataFrame(law_rows),outdir/"table_2_selected_modelB_laws_FULL.csv")
    table_export = write_table2_publication_bundle(
        df, all_df, pd.DataFrame(law_rows), outdir, alphas, etas)
    (outdir/"braess_modelB_direct_eta_print_table.txt").write_text(df.to_string(index=False),encoding="utf-8")
    export_input_parameters(inst,paths,cfg0,outdir)
    summary=write_validation_summary(full,outdir,{"number_of_parameter_cases":len(rows),"paper_table":"Table 2"})
    return {"table":df,"all_results":all_df,"summary":summary,"output_dir":outdir,
            "table_export_validation":table_export}


def run_experiment_03(outdir):
    """Original representative law plot AND corrected paper Figure 4 alpha grid."""
    import matplotlib.pyplot as plt
    outdir=_prepare_output(outdir);paths=base_path_set();inst=build_route_local_instance();cfg0=SolverConfig()
    b=solve_model_b(inst,paths,cfg0);ev=CostEvaluator(inst,paths,cfg0)
    _,_,tilta=pathwise_costs(ev,b.shares)
    data={"Nominal":inst.probabilities}
    for k in range(paths.n_paths):data[f"Model A P{k+1} at fB"]=tilta[k]
    data["Model B KKT-selected"]=b.tilted_law
    representative=pd.DataFrame(data,index=inst.scenario_names)
    representative.index.name="Scenario"
    representative.to_csv(outdir/"braess_tail_tilt_laws.csv",float_format="%.17g")
    fig,ax=plt.subplots(figsize=(12,5))
    x=np.arange(len(inst.scenario_names));width=.15
    for j,key in enumerate(data):ax.bar(x+(j-2)*width,data[key],width,label=key)
    ax.set_xticks(x,inst.scenario_names,rotation=15,ha='right')
    ax.set_ylabel("Effective scenario probability")
    ax.set_title(r"Representative laws at the same Model B flow: $\alpha=0.90$, $\eta=0.50$")
    ax.legend(loc='upper left',bbox_to_anchor=(1.01,1))
    _save_figure(fig,outdir/"braess_tail_tilt_laws.png")
    alphas=[round(float(a),2) for a in np.arange(.80,.981,.02)]
    matrices=[];long=[];records=[result_record(b)]
    for alpha in alphas:
        cfg=replace(cfg0,alpha=alpha,eta=1.)
        bb=solve_model_b(inst,paths,cfg);ev=CostEvaluator(inst,paths,cfg)
        _,ta,_=pathwise_costs(ev,bb.shares)
        stack=np.vstack([ta,bb.tail_law])
        matrices.append(stack.ravel());records.append(result_record(bb))
        for group in range(4):
            label=f"Model A P{group+1} at fB" if group<3 else "Model B KKT-selected"
            for j,scenario in enumerate(inst.scenario_names):
                long.append({"alpha":alpha,"eta":1.,"Law":label,"Scenario":scenario,
                             "PureTailProbability":float(stack[group,j]),"pi_B":bb.pi,
                             "f1_B":bb.flows[0],"f2_B":bb.flows[1],"f3_B":bb.flows[2]})
    heat=np.column_stack(matrices)
    labels=[f"A: P{k+1} | {s}" for k in range(3) for s in inst.scenario_names]+[f"B: common | {s}" for s in inst.scenario_names]
    fig,ax=plt.subplots(figsize=(11.8,9.5))
    im=ax.imshow(heat,aspect='auto',vmin=0,vmax=1,alpha=.65)
    ax.set_xticks(range(len(alphas)),[f"{a:.2f}" for a in alphas]);ax.set_yticks(range(len(labels)),labels)
    ax.set_xlabel(r"CVaR confidence level $\alpha$")
    ax.set_title(r"Pure CVaR tail laws at each Model B equilibrium ($\eta=1$)"+"\nModel B uses a KKT-compatible tail allocation at ties")
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            if heat[i,j]>.005:ax.text(j,i,f"{heat[i,j]:.2f}",ha='center',va='center',fontsize=7)
    fig.colorbar(im,ax=ax,label='Pure CVaR tail probability',fraction=.03,pad=.02)
    _save_figure(fig,outdir/"figure_4_pure_tail_laws_KKT_selected_600dpi.png")
    dataframe_to_csv(pd.DataFrame(long),outdir/"figure_4_tail_laws_SOURCE_FULL.csv")
    metric_rows=[]
    for instance in [build_common_tail_instance(),inst]:
        a=solve_model_a(instance,paths,cfg0);bb=solve_model_b(instance,paths,cfg0)
        record=ab_metrics(instance,paths,cfg0,a,bb);record['Instance']=instance.name
        metric_rows.append(record);records.extend([result_record(a),result_record(bb)])
    metrics=pd.DataFrame(metric_rows)
    dataframe_to_csv(metrics,outdir/"braess_representative_modelA_modelB_metrics.csv")
    summary=write_validation_summary(records,outdir,{"paper_figure":"Figure 4","number_of_alpha_grid_cases":len(alphas)})
    return {"representative":representative,"alpha_grid":pd.DataFrame(long),"metrics":metrics,"summary":summary,"output_dir":outdir}



### Common-tail and warm-start experiment (notebook 04)

In [8]:
# Common-tail and warm-start experiment (notebook 04)


def run_experiment_04(outdir):
    import matplotlib.pyplot as plt
    outdir=_prepare_output(outdir);figdir=_prepare_output(outdir/'figures')
    cfg0=SolverConfig();paths=base_path_set();common=build_common_tail_instance();local=build_route_local_instance()
    records=[];representatives=[];common_rows=[]
    for inst in [common,local]:
        a=solve_model_a(inst,paths,cfg0);b=solve_model_b(inst,paths,cfg0)
        rr=ab_metrics(inst,paths,cfg0,a,b);rr['Instance']=inst.name;representatives.append(rr)
        records.extend([result_record(a),result_record(b)])
    # Stronger than the old representative-only comparison: both models are
    # solved independently on the full retained nominal grid.
    for alpha,eta in itertools.product(TABLE_ALPHAS,TABLE_ETAS):
        cfg=replace(cfg0,alpha=alpha,eta=eta)
        a=solve_model_a(common,paths,cfg);b=solve_model_b(common,paths,cfg)
        rr=ab_metrics(common,paths,cfg,a,b)
        if rr['D_f_AB']>2e-7 or abs(a.pi-b.pi)>2e-6:
            raise NumericalValidationError(f"Common-tail equivalence check failed: {rr}")
        common_rows.append(rr);records.extend([result_record(a),result_record(b)])
    rows=[];summary_rows=[]
    warm_alphas=[.80,.84,.88,.92];warm_etas=[round(i/10,1) for i in range(1,11)]
    for alpha,eta in itertools.product(warm_alphas,warm_etas):
        cfg=replace(cfg0,alpha=alpha,eta=eta)
        b=solve_model_b(local,paths,cfg);records.append(result_record(b))
        au=solve_model_a(local,paths,cfg)
        aw=solve_model_a(local,paths,cfg,initial_flows=b.flows)
        if np.linalg.norm(au.flows-aw.flows,ord=np.inf)>2e-7:
            raise NumericalValidationError("The two initializations returned different Model A equilibria.")
        for label,a in [('Uniform initialization',au),('Model B warm start',aw)]:
            records.append(result_record(a))
            row={"alpha":alpha,"eta":eta,"Initialization":label,
                 "Iterations":a.iterations,"CPUSeconds":a.elapsed_seconds,
                 "FinalResidual":a.checks['fixed_point_relative']*cfg.q,
                 "Gap_A":a.checks['vi_gap_relative'],"pi_A":a.pi,
                 "ActiveCount_A":a.active_count,"pi_B":b.pi,
                 "ModelBIterations":b.iterations,"ModelBCPUSeconds":b.elapsed_seconds,
                 "KKT_B":b.checks['kkt_stationarity']}
            for k in range(paths.n_paths):
                row[f"f{k+1}_A_over_q"]=a.shares[k];row[f"f{k+1}_B_over_q"]=b.shares[k]
            rows.append(row)
        summary_rows.append({"alpha":alpha,"eta":eta,"Iterations_uniform":au.iterations,
                             "Iterations_warm":aw.iterations,"Iteration_reduction":au.iterations-aw.iterations,
                             "Iteration_reduction_pct":100*(au.iterations-aw.iterations)/au.iterations,
                             "Gap_A_uniform":au.checks['vi_gap_relative'],"Gap_A_warm":aw.checks['vi_gap_relative'],
                             "FinalResidual_uniform":au.checks['fixed_point_relative']*cfg.q,
                             "FinalResidual_warm":aw.checks['fixed_point_relative']*cfg.q,
                             "pi_A_uniform":au.pi,"pi_A_warm":aw.pi,"pi_B":b.pi,
                             "CPU_uniform":au.elapsed_seconds,"CPU_warm_ModelA_only":aw.elapsed_seconds,
                             "CPU_ModelB":b.elapsed_seconds,"CPU_warm_total":b.elapsed_seconds+aw.elapsed_seconds,
                             "ModelA_flow_agreement_inf":float(np.max(np.abs(au.flows-aw.flows)))})
    long=pd.DataFrame(rows);summary=pd.DataFrame(summary_rows);common_df=pd.DataFrame(common_rows)
    dataframe_to_csv(pd.DataFrame(representatives),outdir/'braess_modelA_modelB_exactness_diagnostics.csv')
    dataframe_to_csv(common_df,outdir/'common_tail_full_grid_independent_checks.csv')
    dataframe_to_csv(long,outdir/'braess_modelA_warm_start_direct_eta_long.csv')
    dataframe_to_csv(summary,outdir/'braess_modelA_warm_start_direct_eta_iterations_only.csv')
    for alpha in warm_alphas:
        sub=summary[summary.alpha.eq(alpha)].sort_values('eta')
        fig,ax=plt.subplots(figsize=(8,4.3))
        ax.plot(sub.eta,sub.Iterations_uniform,marker='o',label='Uniform initialization')
        ax.plot(sub.eta,sub.Iterations_warm,marker='s',label='Model B warm start')
        ax.set_xlabel(r"CVaR weight $\eta$");ax.set_ylabel('Model A fixed-point iterations')
        ax.set_title(rf"Model A convergence, $\alpha={alpha:.2f}$")
        ax.set_xticks(warm_etas);ax.legend();ax.grid(True,alpha=.25)
        _save_figure(fig,figdir/f"warm_start_alpha_{alpha:.2f}_600dpi.png")
    spec=[('alpha',r'$\alpha$','.2f'),('eta',r'$\eta$','.1f'),
          ('Iterations_uniform','Iter. uniform','.0f'),('Iterations_warm','Iter. warm','.0f'),
          ('Iteration_reduction',r'$\Delta$ iter.','.0f'),
          ('Iteration_reduction_pct','Iter. reduction (percent)','.1f'),('Gap_A_warm',r'$\mathrm{Gap}_A$ warm','.3e')]
    write_latex_table(summary,outdir/'braess_modelA_warm_start_direct_eta_table.tex',spec,
                     r"Model A warm-start iterations on the original direct risk grid. Model B is solved and certified independently before it is used as an initialization.",
                     'tab:braess_modelA_warm_start_direct_eta')
    agg=summary.groupby('alpha').agg(n_eta=('eta','size'),mean_iter_uniform=('Iterations_uniform','mean'),
                                   mean_iter_warm=('Iterations_warm','mean'),mean_iter_saved=('Iteration_reduction','mean'),
                                   mean_iter_reduction_pct=('Iteration_reduction_pct','mean'),max_gap_warm=('Gap_A_warm','max')).reset_index()
    dataframe_to_csv(agg,outdir/'braess_modelA_warm_start_direct_eta_alpha_summary_iterations_only.csv')
    spec=[('alpha',r'$\alpha$','.2f'),('n_eta',r'$|\eta|$','.0f'),
          ('mean_iter_uniform','Mean iter. uniform','.1f'),('mean_iter_warm','Mean iter. warm','.1f'),
          ('mean_iter_saved','Mean iter. saved','.1f'),('mean_iter_reduction_pct','Mean reduction (percent)','.1f'),
          ('max_gap_warm',r'Max $\mathrm{Gap}_A$','.3e')]
    write_latex_table(agg,outdir/'braess_modelA_warm_start_direct_eta_summary_table.tex',spec,
                     r"Iteration counts for the two Model A initializations. The complete CSV also reports the extra Model B solve time; fewer Model A iterations alone are not a total-time speedup certificate.",
                     'tab:braess_warm_start')
    validation=write_validation_summary(records,outdir,{'common_tail_cases':len(common_rows),'warm_start_parameter_cases':40,
                                                       'independent_ModelA_warm_start_runs':80})
    return {'summary_table':summary,'common_tail':common_df,'alpha_summary':agg,'summary':validation,'output_dir':outdir}



### Full SP/DRO before-after grid and single-source tables (notebook 06)

In [9]:
# Full SP/DRO before-after grid and single-source tables (notebook 06)


def run_experiment_06(outdir):
    outdir=_prepare_output(outdir);cfg0=SolverConfig();paths=base_path_set()
    before=build_route_local_instance();after=build_reliable_bc_upgrade(before);D=scenario_distance_matrix()
    records=[];results={};plans=[]
    alphas=[.92,.96];etas=TABLE_ETAS;rhos=[0.,.05,.10,.20]
    # Serial execution is intentional: no fork, process-pool fallback, hidden
    # state, or notebook-specific multiprocessing behavior is required.
    for network,alpha,eta,rho in itertools.product(['Before','After'],alphas,etas,rhos):
        inst=before if network=='Before' else after
        cfg=replace(cfg0,alpha=alpha,eta=eta)
        r=solve_model_b(inst,paths,cfg,center=TRAIN_LAW,rho=rho,distances=D)
        model='Empirical Model B-SP' if rho==0 else 'Wasserstein DRO Model B'
        row=result_record(r,rho,network);row['Model']=model
        row.update(evaluate_test(inst,paths,r))
        records.append(row);results[(network,alpha,eta,rho)]=r
        if r.transport is not None:
            for ii,i in enumerate(np.flatnonzero(TRAIN_LAW>0)):
                for j in range(len(TRAIN_LAW)):
                    plans.append({'Network':network,'alpha':alpha,'eta':eta,'rho':rho,
                                  'FromScenario':before.scenario_names[i],'ToScenario':before.scenario_names[j],
                                  'Mass':r.transport[ii,j],'Distance':D[i,j],
                                  'TransportCost':r.transport[ii,j]*D[i,j]})
    all_df=pd.DataFrame(records).sort_values(['alpha','eta','rho','Network']).reset_index(drop=True)
    if len(all_df)!=96 or not all_df.passed.eq(True).all():
        raise NumericalValidationError('The full 96-case SP/DRO grid was not validated.')
    direct=all_df[(all_df.Network=='Before')&np.isclose(all_df.rho,.10)].copy()
    radius=all_df[(all_df.Network=='Before')&np.isclose(all_df.alpha,.96)&(all_df.rho>0)].sort_values(['rho','eta']).copy()
    before_compare=all_df[(all_df.Network=='Before')&(all_df.rho.isin([0.,.10]))].copy()
    pair_rows=[];decision_rows=[]
    for alpha,eta in itertools.product(alphas,etas):
        row={'RiskCase':risk_case(eta),'alpha':alpha,'eta':eta}
        for rho in rhos:
            rb=all_df[(all_df.Network=='Before')&all_df.alpha.eq(alpha)&all_df.eta.eq(eta)&all_df.rho.eq(rho)].iloc[0]
            ra=all_df[(all_df.Network=='After')&all_df.alpha.eq(alpha)&all_df.eta.eq(eta)&all_df.rho.eq(rho)].iloc[0]
            delta=float(ra.CETest-rb.CETest)
            pair_rows.append({'RiskCase':risk_case(eta),'alpha':alpha,'eta':eta,'rho':rho,'Model':rb.Model,
                              'P3Before':rb.P3,'P3After':ra.P3,'piBefore':rb.pi_od,'piAfter':ra.pi_od,
                              'MeanTestBefore':rb.MeanTest,'MeanTestAfter':ra.MeanTest,
                              'CVaRTestBefore':rb.CVaRTest,'CVaRTestAfter':ra.CVaRTest,
                              'CETestBefore':rb.CETest,'CETestAfter':ra.CETest,
                              'DeltaMeanTest':float(ra.MeanTest-rb.MeanTest),
                              'DeltaCVaRTest':float(ra.CVaRTest-rb.CVaRTest),'DeltaCETest':delta,
                              'MeanTravelTimeBefore':rb.MeanTravelTimeTest,'MeanTravelTimeAfter':ra.MeanTravelTimeTest,
                              'DecisionByCE':'Improve' if delta<0 else 'Do not improve'})
            if rho==0:
                row.update({'P3_SP_Before':rb.P3,'P3_SP_After':ra.P3,'DeltaCE_SP':delta})
            else:row[f'DeltaCE_DRO_{rho:.2f}']=delta
        ds=[row[k] for k in row if k.startswith('DeltaCE')]
        row['CE_decision']='Improve' if all(v<0 for v in ds) else ('Do not improve' if all(v>0 for v in ds) else 'Model-dependent')
        decision_rows.append(row)
    decision=pd.DataFrame(decision_rows);pair=pd.DataFrame(pair_rows)
    # Table 7 is a FILTER of the same master table. It NEVER solves or recovers
    # another distribution and therefore cannot silently contradict Tables 4/5.
    empirical=all_df[(all_df.Network=='Before')&all_df.alpha.eq(.96)&all_df.eta.eq(.6)&all_df.rho.eq(0)].iloc[0].copy()
    empirical['Network']='Before/After'
    worst_rows=[empirical.to_dict()]
    for rho,network in itertools.product([.05,.1,.2],['Before','After']):
        rr=all_df[(all_df.Network==network)&all_df.alpha.eq(.96)&all_df.eta.eq(.6)&all_df.rho.eq(rho)]
        if len(rr)!=1:raise NumericalValidationError('Table 7 key is not unique in the master data.')
        worst_rows.append(rr.iloc[0].to_dict())
    worst=pd.DataFrame(worst_rows)[['Model','Network','rho','alpha','eta']+['QStar'+k for k in SCENARIO_KEYS]].copy()
    # Round-trip numeric consistency is asserted before writing any table.
    for _,r in worst[worst.rho>0].iterrows():
        ref=results[(r.Network,float(r.alpha),float(r.eta),float(r.rho))]
        q=np.array([r['QStar'+k] for k in SCENARIO_KEYS])
        if not np.array_equal(q,ref.raw_law):raise NumericalValidationError('Table 7 no longer uses the selected law of its equilibrium.')
    dataframe_to_csv(all_df,outdir/'00_case_results_master_full.csv')
    dataframe_to_csv(direct,outdir/'01_table_dro_direct_eta_grid_alpha092_096_SOURCE_FULL.csv')
    dataframe_to_csv(radius,outdir/'02_table_dro_alpha096_rho_eta_grid_SOURCE_FULL.csv')
    dataframe_to_csv(before_compare,outdir/'03_table_sp_dro_before_upgrade_direct_grid_SOURCE_FULL.csv')
    dataframe_to_csv(pair,outdir/'04_table_sp_dro_reliable_bc_decision_SOURCE_FULL.csv')
    dataframe_to_csv(all_df,outdir/'05_table_sp_dro_before_after_flows_SOURCE_FULL.csv')
    dataframe_to_csv(worst,outdir/'06_table_sp_dro_worst_case_probabilities_DEDUP_SOURCE_FULL.csv')
    dataframe_to_csv(all_df,outdir/'07_worst_case_probabilities_FULL_GRID.csv')
    dataframe_to_csv(pd.DataFrame(plans),outdir/'11_selected_transport_plans_FULL.csv')
    base_spec=[('RiskCase','Risk case',None),('alpha',r'$\alpha$','.2f'),('eta',r'$\eta$','.1f')]
    flow_spec=[(f'P{k}',rf'$f_{k}^R/q$','.3f') for k in [1,2,3]]
    law_spec=[('pi_od',r'$\pi^R$','.2f'),('ActiveCount',r'$|\mathcal A^R|$','.0f'),
              ('QStarSystemWide',r'$Q^*_{\rm sys}$','.3f'),('RobustTiltShortcut',r'$\widetilde Q^R_{\rm sh}$','.3f'),
              ('RobustTiltSystemWide',r'$\widetilde Q^R_{\rm sys}$','.3f'),('CETest',r'$\mathrm{CE}^{\rm Test}_{\alpha,\eta}$','.1f')]
    write_latex_table(direct,outdir/'table_dro_direct_eta_grid_alpha092_096.tex',base_spec+flow_spec+law_spec,
                     r"Finite-support Wasserstein DRO grid on the before-upgrade network, $\rho=0.10$. All displayed probabilities and reservation costs use the jointly selected risk-optimal, KKT-compatible law.",
                     'tab:braess_dro_direct_eta_grid_full')
    rspec=[('RiskCase','Risk case',None),('rho',r'$\rho$','.2f'),('eta',r'$\eta$','.1f')]+flow_spec+law_spec
    write_latex_table(radius,outdir/'table_dro_alpha096_rho_eta_grid.tex',rspec,
                     r"Finite-support Wasserstein radius sensitivity on the before-upgrade network, $\alpha=0.96$. Raw laws need not be unique; the same selected laws are used throughout the reported tables.",
                     'tab:braess_dro_alpha096_rho_eta_grid')
    dspec=base_spec+[('P3_SP_Before',r'$P_3^{\rm SP,before}$','.3f'),('P3_SP_After',r'$P_3^{\rm SP,after}$','.3f'),
                     ('DeltaCE_SP',r'$\Delta\mathrm{CE}^{\rm SP}$','.1f')]
    dspec += [(f'DeltaCE_DRO_{rho:.2f}',rf'$\Delta\mathrm{{CE}}^{{\rm DRO}}_{{\rho={rho:.2f}}}$','.1f') for rho in [.05,.1,.2]]
    dspec += [('CE_decision','CE decision',None)]
    write_latex_table(decision,outdir/'table_sp_dro_reliable_bc_decision_grid_alpha092_096.tex',dspec,
                     r"Reliable $B\to C$ upgrade evaluation on the retained grid. Negative changes indicate lower held-out mean--CVaR Beckmann potential, not a construction-cost or social-welfare assessment.",
                     'tab:braess_dro_reliable_bc_decision')
    wspec=[('Model','Model',None),('Network','Network',None),('rho',r'$\rho$','.2f')]
    wspec += [('QStar'+k,label,'.3f') for k,label in zip(SCENARIO_KEYS,['Normal','Upper','Lower','Shortcut','System-wide'])]
    write_latex_table(worst,outdir/'table_sp_dro_worst_case_probabilities_dedup.tex',wspec,
                     r"Selected raw worst-case laws at $(\alpha,\eta)=(0.96,0.60)$. These rows are extracted directly from the certified equilibrium master data. Law selection maximizes system-wide mass only within the risk-optimal, KKT-compatible tolerance set.",
                     'tab:braess_dro_worst_case_probs')
    # Original inputs, exported at their full code precision.
    params=[]
    for s in before.scenario_names:
        old,new=before.link_params[s]['BC'],after.link_params[s]['BC']
        params.append({'Scenario':s,'Before_t0_BC':old.t0,'Before_capacity_BC':old.capacity,'Before_delta_BC':old.delay,
                       'After_t0_BC':new.t0,'After_capacity_BC':new.capacity,'After_delta_BC':new.delay})
    dataframe_to_csv(pd.DataFrame(params),outdir/'08_bc_upgrade_parameters.csv')
    pspec=[('Scenario','State',None),('Before_t0_BC',r'$t^0_{BC}$ before','.2f'),
           ('Before_capacity_BC',r'$c_{BC}$ before','.1f'),('Before_delta_BC',r'$\Delta_{BC}$ before','.1f'),
           ('After_t0_BC',r'$t^0_{BC}$ after','.2f'),('After_capacity_BC',r'$c_{BC}$ after','.1f'),
           ('After_delta_BC',r'$\Delta_{BC}$ after','.1f')]
    write_latex_table(pd.DataFrame(params),outdir/'table_bc_upgrade_time_functions.tex',pspec,
                     r'Before--after parameters for the reliable $B\to C$ upgrade. Unrounded input values are retained in the source CSV.',
                     'tab:braess_dro_upgrade_bc_functions')
    bspec=base_spec+[('Model','Model',None),('rho',r'$\rho$','.2f')]+flow_spec+law_spec
    write_latex_table(before_compare,outdir/'table_sp_dro_before_upgrade_direct_grid_alpha092_096.tex',bspec,
                     r'Empirical SP and Wasserstein DRO equilibria before the upgrade. All laws and costs are validated at their returned flows.',
                     'tab:braess_sp_dro_direct_grid')
    dataframe_to_csv(pd.DataFrame({'Scenario':before.scenario_names,'TrainingCount':TRAIN_COUNTS,
                                  'TrainingLaw':TRAIN_LAW,'HeldOutTestLaw':TEST_LAW,'NominalBraessLaw':before.probabilities}),
                     outdir/'09_training_test_laws.csv')
    pd.DataFrame(D,index=before.scenario_names,columns=before.scenario_names).to_csv(outdir/'10_scenario_distance_matrix.csv',float_format='%.17g')
    export_input_parameters(before,paths,cfg0,outdir)
    summary=write_validation_summary(records,outdir,{'number_of_parameter_cases':96,'tables_4_5_7_share_one_master':True})
    return {'master':all_df,'worst_case_table':worst,'decision_table':decision,'summary':summary,'output_dir':outdir,'results':results}



### Regression tests and independent formulation checks

In [10]:
# Regression tests and independent formulation checks


def run_regression_tests(outdir):
    """Executable regressions for the two reported errors and edge cases."""
    outdir=_prepare_output(outdir)
    checks=[]
    def check(name,condition,value,tolerance,description):
        if not bool(condition):
            raise NumericalValidationError(f"Regression failed: {name}; value={value}; tolerance={tolerance}")
        checks.append({'Test':name,'Passed':True,'Observed':float(value),'Tolerance':float(tolerance),'Description':description})
    paths=base_path_set();inst=build_route_local_instance();cfg0=SolverConfig();rng=np.random.default_rng(20260922)
    # Manual scalar formulas are independent of the vectorized evaluator.
    max_cost_error=max_potential_error=max_gradient_error=max_entropy_error=0.
    for net in [inst,build_common_tail_instance(),build_reliable_bc_upgrade(inst)]:
        ev=CostEvaluator(net,paths,cfg0)
        for _ in range(8):
            y=rng.dirichlet(np.ones(paths.n_paths)+2);ell,T,times,x=ev.values(y)
            for s_idx,s in enumerate(net.scenario_names):
                manual_L=0.;manual_t=[]
                for j,link in enumerate(LINKS):
                    par=net.link_params[s][link];z=x[j];b=cfg0.bpr_beta;g=cfg0.bpr_gamma
                    t=par.t0*(1+b*(z/par.capacity)**g)+par.delay
                    integral=par.t0*(z+b*z**(g+1)/((g+1)*par.capacity**g))+par.delay*z
                    manual_t.append(t);manual_L+=integral
                max_cost_error=max(max_cost_error,float(np.max(np.abs(np.asarray(manual_t)-times[s_idx]))))
                max_potential_error=max(max_potential_error,abs(manual_L/cfg0.q-ell[s_idx]))
            h=2e-6
            for k in range(paths.n_paths):
                v=np.zeros(paths.n_paths);v[k]=h
                deriv=(ev.values(y+v)[0]-ev.values(y-v)[0])/(2*h)
                max_gradient_error=max(max_gradient_error,float(np.max(np.abs(deriv-T[:,k]))))
                derivH=(ev.regularizer(y+v)[0]-ev.regularizer(y-v)[0])/(2*h)
                max_entropy_error=max(max_entropy_error,abs(derivH-ev.regularizer(y)[1][k]))
    check('Original scalar BPR costs',max_cost_error<1e-10,max_cost_error,1e-10,'Vectorized values equal the original per-link equations.')
    check('Canonical potential and flow scaling',max_potential_error<1e-10,max_potential_error,1e-10,'No rounded paper inputs; zero-normalized BPR integrals divided by q.')
    check('Analytic potential gradient',max_gradient_error<2e-7,max_gradient_error,2e-7,'Central differences versus path times d(L/q)/d(f/q).')
    check('Analytic entropy gradient',max_entropy_error<2e-7,max_entropy_error,2e-7,'Central differences versus log(1+f/r0)/theta.')
    cv_error=0.
    for _ in range(100):
        p=rng.dirichlet(np.ones(5));p[rng.integers(0,5)]=0.;p=p/p.sum()
        v=rng.normal(size=5)*10
        if rng.random()<.5:v[2]=v[1]
        a=float(rng.choice([.1,.5,.8,.92,.96,.99]))
        cv_error=max(cv_error,abs(finite_cvar(v,p,a)-ru_cvar_independent(v,p,a)))
    check('Zero probabilities, atoms, and tied CVaR values',cv_error<1e-9,cv_error,1e-9,'100 independently evaluated finite CVaR/RU cases.')
    # Analytically solvable two-state ambiguity example.
    env=FiniteRiskEnvelope([1.,0.],.8,.5,.1,np.array([[0.,1.],[1.,0.]]))
    risk,_=env.maximize(np.array([0.,10.]))
    check('Analytic two-state mean-CVaR DRO',abs(risk-3.)<1e-10,abs(risk-3.),1e-10,'Worst mean=1, CVaR=5, eta=.5 gives risk=3.')
    env=FiniteRiskEnvelope([1.,0.],.95,1.,.1,np.array([[0.,1.],[1.,0.]]))
    risk,res=env.maximize(np.array([0.,10.]));q,v,w,plan=env.unpack(res.x)
    check('Pure-CVaR fixed-threshold degeneracy',abs(risk-10.)<1e-10 and q[1]>=.05-1e-10,
          abs(risk-10.),1e-10,'Joint envelope cannot substitute the empirical law when a maximum threshold makes all hinges zero.')
    # Targeted Table 2 regression using full-precision notebook data.
    cfg=replace(cfg0,alpha=.80,eta=1.)
    b=solve_model_b(inst,paths,cfg);ev=CostEvaluator(inst,paths,cfg)
    l,T,_,_=ev.values(b.shares);H,dH=ev.regularizer(b.shares)
    naive_tail=finite_tail_law(l,inst.probabilities,cfg.alpha)
    naive_F=T.T@naive_tail+dH
    naive_pi,_=reservation_and_target(T.T@naive_tail,cfg)
    check('Table 2 corrected pi, alpha=.80 eta=1',abs(b.pi-32.572827954)<2e-6,abs(b.pi-32.572827954),2e-6,
          'Reference uses original code precision, not rounded paper link parameters.')
    check('Canonical sorted selector is not a KKT certificate',np.ptp(naive_F)>1.,np.ptp(naive_F),1.,
          'The canonical value-only selector creates a large active-path stationarity spread at the tied optimum.')
    pi_comparison=pd.DataFrame([{'Method':'Canonical sort after solve (invalid equilibrium diagnostics)','pi':naive_pi,
                                 'F1':naive_F[0],'F2':naive_F[1],'F3':naive_F[2],'Spread':np.ptp(naive_F)},
                                {'Method':'Joint risk/KKT selection (corrected)','pi':b.pi,
                                 'F1':b.path_costs[0]+dH[0],'F2':b.path_costs[1]+dH[1],'F3':b.path_costs[2]+dH[2],
                                 'Spread':np.ptp(b.path_costs+dH)}])
    # Targeted Table 7 regression: prove the old PDF law is not worst-case.
    cfg=replace(cfg0,alpha=.96,eta=.6);D=scenario_distance_matrix()
    b=solve_model_b(inst,paths,cfg,center=TRAIN_LAW,rho=.05,distances=D)
    ev=CostEvaluator(inst,paths,cfg);l,T,_,_=ev.values(b.shares)
    bad=TRAIN_LAW.copy();moved=.05/D[0,4];bad[0]-=moved;bad[4]+=moved
    alt=TRAIN_LAW.copy();moved=.05/D[3,4];alt[3]-=moved;alt[4]+=moved
    def risk_q(q):return cfg.q*((1-cfg.eta)*float(q@l)+cfg.eta*ru_cvar_independent(l,q,cfg.alpha))
    bad_gap=b.risk_value-risk_q(bad);alt_gain=risk_q(alt)-risk_q(bad)
    check('Reject the old PDF Table 7 law',bad_gap>50.,bad_gap,50.,'A feasible nominal-to-system-wide transfer does not maximize the full risk objective.')
    check('Independent feasible counterexample',alt_gain>50.,alt_gain,50.,'Shortcut-to-system-wide transfer beats the old PDF law under the same transport budget.')
    check('Table 7 selected systemic mass',abs(b.raw_law[-1]-.04)<2e-7,abs(b.raw_law[-1]-.04),2e-7,
          'The optimal joint raw law need not maximize systemic mass without risk/KKT constraints.')
    law_comparison=[]
    for label,q in [('Old PDF law',bad),('Feasible counterexample',alt),('Corrected selected law',b.raw_law)]:
        row={'Law':label,'RiskValue':risk_q(q),'GapToWorstRisk':b.risk_value-risk_q(q),
             'WassersteinDistance':minimum_transport_cost(TRAIN_LAW,q,D)}
        row.update({key:float(x) for key,x in zip(SCENARIO_KEYS,q)});law_comparison.append(row)
    # rho=0 must be exactly the empirical SP model, independently through both APIs.
    empir=instance_with_probabilities(inst,TRAIN_LAW,'Empirical test instance')
    a0=solve_model_b(empir,paths,cfg)
    b0=solve_model_b(inst,paths,cfg,center=TRAIN_LAW,rho=0,distances=D)
    diff=float(np.max(np.abs(a0.flows-b0.flows)))
    check('rho=0 reduces to empirical Model B',diff<1e-8,diff,1e-8,'Same empirical law, no radius reinterpretation.')
    # The validator must reject a feasible but non-equilibrium flow.
    perturbed=b.shares.copy();perturbed[0]+=.02;perturbed[2]-=.02
    l,T,_,_=ev.values(perturbed);H,dH=ev.regularizer(perturbed)
    env=FiniteRiskEnvelope(TRAIN_LAW,cfg.alpha,cfg.eta,.05,D)
    selection=select_kkt_law(env,l,T,dH,perturbed)
    _,_,diag,fail=certify_model_b(ev,perturbed,env,selection,selection['risk_max']+H,0.)
    check('Reject a perturbed flow even with optimal risk value',bool(fail),diag['kkt_stationarity'],TOL.kkt,
          'Optimal adversarial value alone is insufficient for equilibrium consistency.')
    df=pd.DataFrame(checks)
    dataframe_to_csv(df,outdir/'REGRESSION_TESTS.csv')
    dataframe_to_csv(pi_comparison,outdir/'table_2_pi_bug_regression.csv')
    dataframe_to_csv(pd.DataFrame(law_comparison),outdir/'table_7_law_bug_regression.csv')
    (outdir/'REGRESSION_SUMMARY.json').write_text(json.dumps({'all_passed':True,'tests':len(df),'core_version':CORE_VERSION},indent=2))
    return {'tests':df,'pi_comparison':pi_comparison,'law_comparison':pd.DataFrame(law_comparison),'output_dir':outdir}


## Output location and manuscript target

`PAPER_TABLE_PATH` is the exact standalone `.tex` table that will be updated after successful
validation. Change only this path to point to the table used by your LaTeX project. Any old copy
of that table is backed up before replacement. Other `.tex` files are not searched or modified.

In [11]:
from datetime import datetime, timezone
from IPython.display import display, FileLink
import zipfile

# Set this to the actual standalone Table 2 file used by your LaTeX project.
# For example: Path("tables") / "braess_modelB_direct_eta_table_full.tex"
PAPER_TABLE_PATH = Path("braess_modelB_direct_eta_table_full.tex")

RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
OUTPUT_ROOT = Path("verified_braess_outputs") / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
print("Run output directory:", OUTPUT_ROOT.resolve())
print("Manuscript table to update:", PAPER_TABLE_PATH.resolve())
print("Numerical core:", CORE_VERSION)
print("Table export:", TABLE2_EXPORT_VERSION)
print("Python:", platform.python_version(), "SciPy:", scipy.__version__, "NumPy:", np.__version__)
print("Acceptance tolerances:", vars(TOL))


Run output directory: /mnt/data/table2_synced_delivery/verified_braess_outputs/run_20260923T041754_604925Z
Manuscript table to update: /mnt/data/table2_synced_delivery/braess_modelB_direct_eta_table_full.tex
Numerical core: braess-kkt-joint-risk-envelope-v1
Table export: table2-single-source-sync-v2
Python: 3.13.5 SciPy: 1.17.0 NumPy: 2.3.5
Acceptance tolerances: {'probability': 2e-09, 'flow_feasibility': 2e-08, 'kkt': 1e-06, 'kernel_relative': 2e-07, 'risk_per_demand': 5e-08, 'global_gap_per_demand': 1e-06, 'a_vi_relative': 2e-09, 'a_fixed_point_relative': 2e-09}


## Mandatory regression tests

These tests include both reported failures, zero probabilities, tied CVaR values, `eta=0`, `eta=1`, `rho=0`, and an intentionally perturbed non-equilibrium flow.

In [12]:
regression = run_regression_tests(OUTPUT_ROOT / "00_regressions")
display(regression["tests"])
display(regression["pi_comparison"])
display(regression["law_comparison"])


,Test,Passed,Observed,Tolerance,Description
0,Original scalar BPR costs,True,7.105427e-15,1.000000e-10,Vectorized values equal the original per-link ...
1,Canonical potential and flow scaling,True,7.105427e-15,1.000000e-10,No rounded paper inputs; zero-normalized BPR i...
2,Analytic potential gradient,True,3.569220e-09,2.000000e-07,Central differences versus path times d(L/q)/d...
3,Analytic entropy gradient,True,5.123875e-10,2.000000e-07,Central differences versus log(1+f/r0)/theta.
4,"Zero probabilities, atoms, and tied CVaR values",True,3.552714e-15,1.000000e-09,100 independently evaluated finite CVaR/RU cases.
5,Analytic two-state mean-CVaR DRO,True,4.440892e-16,1.000000e-10,"Worst mean=1, CVaR=5, eta=.5 gives risk=3."
6,Pure-CVaR fixed-threshold degeneracy,True,0.000000e+00,1.000000e-10,Joint envelope cannot substitute the empirical...
7,"Table 2 corrected pi, alpha=.80 eta=1",True,5.409362e-10,2.000000e-06,"Reference uses original code precision, not ro..."
8,Canonical sorted selector is not a KKT certifi...,True,1.848815e+01,1.000000e+00,The canonical value-only selector creates a la...
9,Reject the old PDF Table 7 law,True,1.135168e+02,5.000000e+01,A feasible nominal-to-system-wide transfer doe...


,Method,pi,F1,F2,F3,Spread
0,Canonical sort after solve (invalid equilibriu...,28.009714,45.589167,27.101022,28.127840,1.848815e+01
1,Joint risk/KKT selection (corrected),32.572828,32.572828,32.572828,32.572828,7.105427e-15


,Law,RiskValue,GapToWorstRisk,WassersteinDistance,Normal,Upper,Lower,Shortcut,SystemWide
0,Old PDF law,2471.798057,1.135168e+02,0.05,0.511493,0.181818,0.181818,0.090909,0.033962
1,Feasible counterexample,2585.089207,2.256028e-01,0.05,0.545455,0.181818,0.181818,0.049635,0.041274
2,Corrected selected law,2585.314810,5.330548e-08,0.05,0.539536,0.181818,0.181818,0.056828,0.040000


## Recompute all 30 cases, audit all table cells, and update the paper table

This cell also updates `PAPER_TABLE_PATH`; it does not leave the manuscript copy disconnected
from the newly generated results. No publication occurs when a numerical or export check fails.

In [13]:
out01 = run_experiment_01(OUTPUT_ROOT / "01_table2")
export_regressions = run_table2_export_regressions(out01)
publication = publish_table2_outputs(out01, PAPER_TABLE_PATH)

preview = ["alpha", "eta", "f1_A_over_q", "f2_A_over_q", "f3_A_over_q",
           "f1_B_over_q", "f2_B_over_q", "f3_B_over_q", "pi_A", "pi_B",
           "D_tail_AB", "D_p_AB", "D_f_AB", "Gap_A_at_fB", "KKT_B"]
display(out01["table"][preview])
display(export_regressions)
print("NUMERICAL CHECKS")
print(json.dumps(out01["summary"], indent=2))
print("TABLE SOURCE AND EXPORT CHECKS")
print(json.dumps(out01["table_export_validation"], indent=2))
print("MANUSCRIPT TABLE UPDATED:", publication["target"])
if publication["backup"] is not None:
    print("Previous copy backed up at:", publication["backup"])
print("Checked rows:", publication["report"]["rows_checked"],
      "| numeric cells:", publication["report"]["numeric_cells_checked"],
      "| manuscript/run files identical:", publication["report"]["publication_matches_checked_run_byte_for_byte"])
for path in [publication["target"], publication["csv_path"],
             out01["output_dir"] / "TABLE2_EXPORT_VALIDATION.json",
             out01["output_dir"] / "TABLE2_PUBLICATION_REPORT.json"]:
    display(FileLink(os.path.relpath(path, Path.cwd())))

# This is the exact same LaTeX text written to the paper table, not an old cell output.
print("\nUPDATED MANUSCRIPT LATEX\n")
print(publication["target"].read_text(encoding="utf-8"))


,alpha,eta,f1_A_over_q,f2_A_over_q,f3_A_over_q,f1_B_over_q,f2_B_over_q,f3_B_over_q,pi_A,pi_B,D_tail_AB,D_p_AB,D_f_AB,Gap_A_at_fB,KKT_B
0,0.80,0.0,2.433060e-02,0.146591,0.829078,0.024331,0.146591,0.829078,24.102212,24.102212,1.600000e+00,0.000000e+00,2.266583e-11,0.000000e+00,3.552714e-15
1,0.80,0.2,9.754001e-03,0.129559,0.860687,0.111133,0.183022,0.705845,26.830758,26.150586,1.600000e+00,3.200000e-01,3.096846e-01,2.106981e-02,3.552714e-15
2,0.80,0.4,8.899325e-04,0.113917,0.885193,0.195871,0.275454,0.528676,29.555119,27.908483,1.405214e+00,5.620855e-01,7.130346e-01,6.303556e-02,0.000000e+00
3,0.80,0.6,1.139193e-11,0.099238,0.900762,0.266051,0.384033,0.349916,32.243838,29.507587,1.313371e+00,7.880228e-01,1.101691e+00,1.159784e-01,3.552714e-15
4,0.80,0.8,1.139193e-11,0.086353,0.913647,0.277443,0.401656,0.320901,34.930443,31.039358,1.226502e+00,9.812016e-01,1.185493e+00,1.278326e-01,7.105427e-15
5,0.80,1.0,1.139193e-11,0.075100,0.924900,0.277443,0.401656,0.320901,37.619898,32.572828,1.167023e+00,1.167023e+00,1.207998e+00,1.298503e-01,7.105427e-15
6,0.84,0.0,2.433060e-02,0.146591,0.829078,0.024331,0.146591,0.829078,24.102212,24.102212,1.500000e+00,0.000000e+00,2.266583e-11,0.000000e+00,3.552714e-15
7,0.84,0.2,1.670399e-02,0.165102,0.818194,0.120650,0.249352,0.629998,27.282801,26.482817,1.500000e+00,3.000000e-01,3.763926e-01,2.542723e-02,1.065814e-14
8,0.84,0.4,1.058728e-02,0.183786,0.805627,0.256952,0.369958,0.373090,30.455306,28.351542,1.411963e+00,5.647853e-01,8.650740e-01,8.301871e-02,3.552714e-15
9,0.84,0.6,5.684405e-03,0.202425,0.791891,0.277443,0.401656,0.320901,33.618959,30.073405,1.245550e+00,7.473298e-01,9.419802e-01,9.249189e-02,3.552714e-15


,Test,Passed
0,Reject a changed pi_B before export,True
1,Reject a stale LaTeX pi_B cell,True
2,Reject CSV/solved-source mismatch,True
3,Refuse to overwrite an unrelated LaTeX document,True


NUMERICAL CHECKS
{
  "core_version": "braess-kkt-joint-risk-envelope-v1",
  "all_checks_passed": true,
  "number_of_equilibrium_records": 60,
  "maxima": {
    "demand_error": 1.836752971939859e-10,
    "nonnegative_flow_error": 0.0,
    "vi_gap_relative": 2.480076041840057e-12,
    "vi_gap_absolute": 7.466496754204854e-09,
    "probability_sum_error": 4.440892098500626e-16,
    "probability_nonnegativity_error": 0.0,
    "tail_capacity_violation": 9.922618282587337e-16,
    "tail_value_error_per_demand": 2.3869475285209774e-09,
    "tilt_identity_error": 0.0,
    "inner_risk_gap_per_demand": 8.21223977709451e-10,
    "raw_law_risk_gap_per_demand": 2.0605739337042905e-13,
    "kkt_stationarity": 8.964768483110674e-09,
    "pi_kernel_error": 4.312212809054472e-10,
    "pi_lp_error": 1.973909036223631e-09,
    "kernel_relative_error": 1.9967683950739002e-09,
    "vi_gap_per_demand": 8.517375249539327e-09,
    "global_gap_per_demand": 9.095629138755612e-09,
    "master_objective_error_per

/mnt/data/table2_synced_delivery/braess_modelB_direct_eta_table_full.tex

/mnt/data/table2_synced_delivery/verified_braess_outputs/run_20260923T041754_604925Z/01_table2/table_2_SOURCE_FULL.csv

/mnt/data/table2_synced_delivery/verified_braess_outputs/run_20260923T041754_604925Z/01_table2/TABLE2_EXPORT_VALIDATION.json

/mnt/data/table2_synced_delivery/verified_braess_outputs/run_20260923T041754_604925Z/01_table2/TABLE2_PUBLICATION_REPORT.json


UPDATED MANUSCRIPT LATEX

% Generated from this run's table_2_SOURCE_FULL.csv; do not copy numbers from an older table.
% Model B laws, costs, and pi_B are selected jointly with equilibrium KKT conditions.
% Required manuscript packages: amsmath, booktabs, graphicx.
\begin{table}[htbp]
\centering
\tiny
\setlength{\tabcolsep}{1.45pt}
\renewcommand{\arraystretch}{1.05}
\caption{Direct-$(\alpha,\eta)$ grid with Model A--Model B flow, reservation-cost, tail-law, and VI diagnostics, reported on the retained grid $\alpha\in\{0.80,0.84,0.88,0.92,0.96\}$ and $\eta\in\{0,0.2,0.4,0.6,0.8,1.0\}$. The table reports normalized path flows, $f_i^A/q$ and $f_i^B/q$, and endogenous reservation generalized costs $\pi^A$ and $\pi^B$. Small nonzero $D_f^{AB}$ and $\mathrm{Gap}_A(f^B)$ values are printed in scientific notation rather than rounded to zero. $D_{\rm tail}^{AB}$ is the maximum pathwise $\ell_1$ distance between pure CVaR-tail laws $p_s\chi_s/(1-\alpha)$; $D_p^{AB}$ is the corresponding distan

## Validated output archive

In [14]:
# Archive this run only. No old CSVs, figures, or cached solutions are included.
manifest = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        manifest.append({"file": str(path.relative_to(OUTPUT_ROOT)),
                         "bytes": path.stat().st_size,
                         "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
(OUTPUT_ROOT / "MANIFEST.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
archive_path = OUTPUT_ROOT.with_suffix(".zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(OUTPUT_ROOT)))
print("Completed. Updated manuscript table:", publication["target"])
print("Validated current-run archive:", archive_path.resolve())
display(FileLink(str(archive_path)))


Completed. Updated manuscript table: /mnt/data/table2_synced_delivery/braess_modelB_direct_eta_table_full.tex
Validated current-run archive: /mnt/data/table2_synced_delivery/verified_braess_outputs/run_20260923T041754_604925Z.zip


/mnt/data/table2_synced_delivery/verified_braess_outputs/run_20260923T041754_604925Z.zip